# agent101 — Build Mini Claude Code from Zero

A step-by-step tutorial that teaches you how to build a mini AI coding agent from scratch.

Based on [learn-claude-code](https://github.com/shareAI-lab/learn-claude-code) by shareAI-lab.

---

# S01: The Agent Loop
`[ s01 ] >  s02 > s03 > s04 > s05 > s06 | s07 > s08 > s09 > s10 > s11`

> *"One loop & PowerShell is all you need"* — one tool + one loop = an agent.
>
> **Harness layer**: The loop — the model’s first connection to the real world.

## Problem

A language model can reason about code, but it can’t *touch* the real world — can’t read files, run tests, or check errors. Without a loop, every tool call requires you to manually copy-paste results back. **You become the loop.**

## Solution

```
+--------+      +-------+      +---------+
|  User  | ---> |  LLM  | ---> |  Tool   |
| prompt |      |       |      | execute |
+--------+      +---+---+      +----+----+
                    ^                |
                    |   tool_result  |
                    +----------------+
                    (loop until model stops calling tools)
```

One exit condition controls the entire flow. The loop runs until the model stops calling tools.

### Step 1: Send a message to the LLM with a tool

First, let’s set up the client, define a `powershell` tool, and send a simple message to see how the LLM responds when it has a tool available.

In [101]:
import os
import json
import subprocess
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
)
MODEL = os.getenv("AZURE_OPENAI_DEPLOYMENT")

SYSTEM = f"You are a coding agent at {os.getcwd()}. If you find we need to call tool, never output raw JSON — always call tools via function calling. Be direct, no explanations."

TOOLS = [{
    "type": "function",
    "function": {
        "name": "powershell",
        "description": "Run a PowerShell command.",
        "parameters": {
            "type": "object",
            "properties": {"command": {"type": "string"}},
            "required": ["command"],
        },
    },
}]

USER_MSG = "show current directory"
# Send a test message and inspect the raw response
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": USER_MSG},
    ],
    tools=TOOLS,
)

msg = response.choices[0].message
print("=== Raw Response ===")
print(f"content:    {msg.content}")
print(f"tool_calls: {msg.tool_calls}")
print()
if msg.tool_calls:
    tc = msg.tool_calls[0]
    print("=== First Tool Call ===")
    print(f"id:        {tc.id}")
    print(f"function:  {tc.function.name}")
    print(f"arguments: {tc.function.arguments}")

=== Raw Response ===
content:    {"command":"Get-Location"}
{
  "command": "Get-Location"
}
tool_calls: None



**What happened?** The LLM didn’t reply with text — it replied with a **tool call**. It wants us to run a PowerShell command.

But nobody executed it! The model can’t touch the real world on its own. That’s what the agent loop is for — we need code that:
1. Executes the tool call
2. Sends the result back to the LLM
3. Repeats until the model stops asking for tools

### Step 2: The Tool Handler

A simple function that executes a PowerShell command and returns the output. Includes basic safety checks and a timeout.



In [102]:
def run_powershell(command: str) -> str:
    """Execute a PowerShell command with safety guards."""
    dangerous = ["Remove-Item -Recurse -Force /", "shutdown", "Restart-Computer", "Stop-Computer"]
    if any(d in command for d in dangerous):
        return "Error: Dangerous command blocked"
    try:
        r = subprocess.run(
            ["powershell", "-NoProfile", "-Command", command],
            cwd=os.getcwd(), capture_output=True, text=True, timeout=120,
        )
        out = (r.stdout + r.stderr).strip()
        return out[:50000] if out else "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: Timeout (120s)"
    except (FileNotFoundError, OSError) as e:
        return f"Error: {e}"

# Quick test
print(run_powershell("Write-Output 'Hello from the agent!'"))

Hello from the agent!


### make real tool call
1. **User prompt** becomes the first message
2. **Send messages + tool definitions** to the LLM
3. **Check the response** — if the model didn’t call a tool, we’re done
4. **Execute each tool call**, collect results, append as messages, loop back to step 2

### Step 3: The Agent Loop


Let’s build it piece by piece.

This is the **entire secret** of an AI coding agent — a `while True` loop that:
1. Calls the LLM with the conversation history + tools
2. Appends the assistant’s response
3. If the model didn’t call any tools → **done** (exit)
4. Otherwise, execute each tool call, append results, and **loop back**

In [103]:
TOOL_HANDLERS = {"powershell": run_powershell}

def agent_loop(messages: list):
    """Core agent loop: call LLM, execute tools, repeat until done."""
    turn = 0
    while True:
        turn += 1
        all_messages = [{"role": "system", "content": SYSTEM}] + messages

        # Show what we send to the LLM
        print(f"\n{'='*60}")
        print(f"Turn {turn} - LLM Input ({len(all_messages)} messages):")
        print(f"{'='*60}")
        for j, m in enumerate(all_messages):
            role = m["role"]
            if role == "system":
                print(f"  [{j}] system: {m['content'][:80]}...")
            elif role == "user":
                content = m.get("content", "")
                if isinstance(content, str):
                    print(f"  [{j}] user: {content[:100]}")
                else:
                    print(f"  [{j}] user: (tool results)")
            elif role == "assistant":
                tc = m.get("tool_calls")
                if tc:
                    print(f"  [{j}] assistant: (called {len(tc)} tool(s))")
                else:
                    print(f"  [{j}] assistant: {str(m.get('content', ''))[:100]}")
            elif role == "tool":
                print(f"  [{j}] tool: {m['content'][:80]}")

        # 1. Call the LLM
        response = client.chat.completions.create(
            model=MODEL,
            messages=all_messages,
            tools=TOOLS,
        )
        msg = response.choices[0].message

        # Show what the LLM returned
        print(f"\nTurn {turn} - LLM Output:")
        print(f"  content:    {msg.content}")
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  tool_call:  {tc.function.name}({tc.function.arguments})")
        else:
            print(f"  tool_calls: None (done!)")

        # 2. Append assistant turn
        assistant_msg = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(assistant_msg)

        # 3. If no tool calls, we're done
        if not msg.tool_calls:
            print(f"\n>>> Agent finished in {turn} turn(s)")
            return

        # 4. Execute each tool call, append results
        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            cmd = args.get('command', '')
            print(f"\n  [exec] {name}: {cmd}")
            output = TOOL_HANDLERS[name](**args)
            print(f"  [result] {output[:300]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": output,
            })

### What just happened?

That’s the **entire agent** in under 30 lines of core logic. Let’s trace the flow:

| Step | What happens |
|------|-------------|
| User types a prompt | Added to `messages` as `{"role": "user", ...}` |
| LLM receives messages + tools | Returns either text or tool calls |
| `msg.tool_calls` is not empty | We execute each tool, append results with `role: "tool"` |
| `msg.tool_calls` is empty/None | Model is done — we exit the loop |

The `messages` list grows with each iteration, giving the LLM full context of what it has done so far.

> **Key insight**: Everything else in this course layers on top of this loop — without changing it.

## Try It!

Run the cell below to start an interactive session. Try these prompts:
1. `Create a file called hello.py that prints "Hello, World!"`
2. `List all files in the current directory`
3. `What is the current git branch?`
4. `Create a directory called test_output and write 3 files in it`

Type `q` or `exit` to quit the REPL.

In [104]:
# Interactive REPL — type your prompts, press Enter. Type "q" to quit.
history = []
while True:
    try:
        query = input("s01 >> ")
    except (EOFError, KeyboardInterrupt):
        break
    if query.strip().lower() in ("q", "exit", ""):
        break
    history.append({"role": "user", "content": query})
    agent_loop(history)
    print()

## create helloWorld.txt and add "Hello world" into it

s01 >>  What is the current git branch?



Turn 1 - LLM Input (2 messages):
  [0] system: You are a coding agent at C:\Users\concao\code\agent101. If you find we need to ...
  [1] user: What is the current git branch?

Turn 1 - LLM Output:
  content:    {
  "command": "git rev-parse --abbrev-ref HEAD"
}
  tool_call:  powershell({"command":"git rev-parse --abbrev-ref HEAD"})

  [exec] powershell: git rev-parse --abbrev-ref HEAD
  [result] master

Turn 2 - LLM Input (4 messages):
  [0] system: You are a coding agent at C:\Users\concao\code\agent101. If you find we need to ...
  [1] user: What is the current git branch?
  [2] assistant: (called 1 tool(s))
  [3] tool: master

Turn 2 - LLM Output:
  content:    The current git branch is **master**.
  tool_calls: None (done!)

>>> Agent finished in 2 turn(s)



s01 >>  q


## What Changed

| Component     | Before     | After                          |
|---------------|------------|--------------------------------|
| Agent loop    | (none)     | `while True` + tool_calls check |
| Tools         | (none)     | `powershell` (one tool)        |
| Messages      | (none)     | Accumulating list              |
| Control flow  | (none)     | Exit when `tool_calls` is empty |

---


# S02:Multiple Tools 

`s01 > [ s02 ] s03 > s04 > s05 > s06 | s07 > s08 > s09 > s10 > s11`

> *"Adding a tool means adding one handler"* — the loop stays the same; new tools register into the dispatch map.
>
> **Harness layer**: Tool dispatch — expanding what the model can reach.

The Pi agent (OpenClaw) only has **4 default tools**. Claude Code has **~ 20 default tools**. The number of tools defines what the agent can do — but the loop never changes. Adding a tool = adding one handler + one schema entry.

## Problem

With only `powershell`, the agent shells out for everything. Reading files means `Get-Content` which truncates unpredictably. Writing means `Set-Content` which can clobber files without guardrails. Every shell call is an unconstrained security surface.

Dedicated tools like `read_file` and `write_file` let you enforce **path sandboxing** at the tool level.

The key insight: **adding tools does not require changing the loop.**

## Solution

```
+--------+      +-------+      +---------------------------+
|  User  | ---> |  LLM  | ---> | Tool Dispatch             |
| prompt |      |       |      | {                         |
+--------+      +---+---+      |   powershell: run_ps      |
                    ^           |   read_file:  run_read    |
                    |           |   write_file: run_write   |
                    +-----------+   edit_file:  run_edit    |
                    tool_result | }                         |
                                +---------------------------+
```

The dispatch map is a dict: `{tool_name: handler_function}`. One lookup replaces any if/elif chain.

### Step 1: Path Sandboxing

Before adding file tools, we need safety. `safe_path()` prevents the agent from escaping the workspace directory.

In [40]:
from pathlib import Path

WORKDIR = Path.cwd()

def safe_path(p: str) -> Path:
    """Resolve path and ensure it stays within the workspace."""
    path = (WORKDIR / p).resolve()
    if not path.is_relative_to(WORKDIR):
        raise ValueError(f"Path escapes workspace: {p}")
    return path

# Test: safe paths work
print(safe_path("hello.txt"))

# Test: escaping is blocked
try:
    safe_path("../../etc/passwd")
except ValueError as e:
    print(f"Blocked: {e}")

C:\Users\concao\code\agent101\hello.txt
Blocked: Path escapes workspace: ../../etc/passwd


### Step 2: New Tool Handlers

Three new handlers: `read_file`, `write_file`, `edit_file`. Each uses `safe_path()` for sandboxing.

In [105]:
def run_read(path: str, limit: int = None) -> str:
    """Read file contents, optionally limiting line count."""
    try:
        text = safe_path(path).read_text(encoding="utf-8")
        lines = text.splitlines()
        if limit and limit < len(lines):
            lines = lines[:limit] + [f"... ({len(lines) - limit} more lines)"]
        return "\n".join(lines)[:50000]
    except Exception as e:
        return f"Error: {e}"


def run_write(path: str, content: str) -> str:
    """Write content to a file, creating parent directories if needed."""
    try:
        fp = safe_path(path)
        fp.parent.mkdir(parents=True, exist_ok=True)
        fp.write_text(content, encoding="utf-8")
        return f"Wrote {len(content)} bytes to {path}"
    except Exception as e:
        return f"Error: {e}"


def run_edit(path: str, old_text: str, new_text: str) -> str:
    """Replace exact text in a file (first occurrence only)."""
    try:
        fp = safe_path(path)
        content = fp.read_text(encoding="utf-8")
        if old_text not in content:
            return f"Error: Text not found in {path}"
        fp.write_text(content.replace(old_text, new_text, 1), encoding="utf-8")
        return f"Edited {path}"
    except Exception as e:
        return f"Error: {e}"

# Quick test
print(run_write("test_s02.txt", "Hello from S02!"))
print(run_read("test_s02.txt"))
print(run_edit("test_s02.txt", "S02", "Session 02"))
print(run_read("test_s02.txt"))

Wrote 15 bytes to test_s02.txt
Hello from S02!
Edited test_s02.txt
Hello from Session 02!


### Step 3: Tool Schemas & Dispatch Map

Register all 4 tools. The LLM sees the schemas; the dispatch map routes calls to handlers.

In [106]:
TOOLS = [
    {"type": "function", "function": {
        "name": "powershell",
        "description": "Run a PowerShell command.",
        "parameters": {
            "type": "object",
            "properties": {"command": {"type": "string"}},
            "required": ["command"],
        },
    }},
    {"type": "function", "function": {
        "name": "read_file",
        "description": "Read file contents. Use 'limit' to cap line count.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string"},
                "limit": {"type": "integer"},
            },
            "required": ["path"],
        },
    }},
    {"type": "function", "function": {
        "name": "write_file",
        "description": "Write content to a file (creates parent dirs).",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string"},
                "content": {"type": "string"},
            },
            "required": ["path", "content"],
        },
    }},
    {"type": "function", "function": {
        "name": "edit_file",
        "description": "Replace exact text in a file (first occurrence).",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string"},
                "old_text": {"type": "string"},
                "new_text": {"type": "string"},
            },
            "required": ["path", "old_text", "new_text"],
        },
    }},
]

TOOL_HANDLERS = {
    "powershell": run_powershell,
    "read_file":  lambda **kw: run_read(kw["path"], kw.get("limit")),
    "write_file": lambda **kw: run_write(kw["path"], kw["content"]),
    "edit_file":  lambda **kw: run_edit(kw["path"], kw["old_text"], kw["new_text"]),
}

print(f"Registered {len(TOOL_HANDLERS)} tools: {list(TOOL_HANDLERS.keys())}")

Registered 4 tools: ['powershell', 'read_file', 'write_file', 'edit_file']


### Step 4: The Agent Loop (unchanged!)

The loop is **exactly the same** as S01. The only difference is that `TOOL_HANDLERS` now has 4 entries instead of 1.

This is the core design: the loop is generic, tools are pluggable.

In [107]:
def agent_loop(messages: list):
    """Core agent loop: call LLM, execute tools, repeat until done."""
    turn = 0
    while True:
        turn += 1
        all_messages = [{"role": "system", "content": SYSTEM}] + messages

        # Show what we send to the LLM
        print(f"\n{'='*60}")
        print(f"Turn {turn} - LLM Input ({len(all_messages)} messages):")
        print(f"{'='*60}")
        for j, m in enumerate(all_messages):
            role = m["role"]
            if role == "system":
                print(f"  [{j}] system: {m['content'][:80]}...")
            elif role == "user":
                content = m.get("content", "")
                if isinstance(content, str):
                    print(f"  [{j}] user: {content[:100]}")
                else:
                    print(f"  [{j}] user: (tool results)")
            elif role == "assistant":
                tc = m.get("tool_calls")
                if tc:
                    print(f"  [{j}] assistant: (called {len(tc)} tool(s))")
                else:
                    print(f"  [{j}] assistant: {str(m.get('content', ''))[:100]}")
            elif role == "tool":
                print(f"  [{j}] tool: {m['content'][:80]}")

        # 1. Call the LLM
        response = client.chat.completions.create(
            model=MODEL,
            messages=all_messages,
            tools=TOOLS,
        )
        msg = response.choices[0].message

        # Show what the LLM returned
        print(f"\nTurn {turn} - LLM Output:")
        print(f"  content:    {msg.content}")
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  tool_call:  {tc.function.name}({tc.function.arguments})")
        else:
            print(f"  tool_calls: None (done!)")

        # 2. Append assistant turn
        assistant_msg = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(assistant_msg)

        # 3. If no tool calls, we're done
        if not msg.tool_calls:
            print(f"\n>>> Agent finished in {turn} turn(s)")
            return

        # 4. Execute each tool call, append results
        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            handler = TOOL_HANDLERS.get(name)
            if handler:
                print(f"\n  [exec] {name}: {json.dumps(args)[:100]}")
                output = handler(**args)
            else:
                output = f"Unknown tool: {name}"
            print(f"  [result] {output[:300]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": output,
            })

## Try It!

Now the agent has 4 tools. Try these prompts:
1. `Read the file pyproject.toml`
2. `Create a file called greet.py with a greet(name) function`
3. `Edit greet.py to add a docstring to the function`
4. `Read greet.py to verify the edit worked`

Watch how the LLM picks the right tool for each task — it no longer shells out for everything.

In [108]:
# Interactive REPL — type your prompts, press Enter. Type "q" to quit.
history = []
while True:
    try:
        query = input("s02 >> ")
    except (EOFError, KeyboardInterrupt):
        break
    if query.strip().lower() in ("q", "exit", ""):
        break
    history.append({"role": "user", "content": query})
    agent_loop(history)
    print()

s02 >>  Create a file called greet.py with a greet(name) function



Turn 1 - LLM Input (2 messages):
  [0] system: You are a coding agent at C:\Users\concao\code\agent101. If you find we need to ...
  [1] user: Create a file called greet.py with a greet(name) function

Turn 1 - LLM Output:
  content:    None
  tool_call:  write_file({"path":"C:\\Users\\concao\\code\\agent101\\greet.py","content":"def greet(name):\n    print(f\"Hello, {name}!\")"})

  [exec] write_file: {"path": "C:\\Users\\concao\\code\\agent101\\greet.py", "content": "def greet(name):\n    print(f\"H
  [result] Wrote 45 bytes to C:\Users\concao\code\agent101\greet.py

Turn 2 - LLM Input (4 messages):
  [0] system: You are a coding agent at C:\Users\concao\code\agent101. If you find we need to ...
  [1] user: Create a file called greet.py with a greet(name) function
  [2] assistant: (called 1 tool(s))
  [3] tool: Wrote 45 bytes to C:\Users\concao\code\agent101\greet.py

Turn 2 - LLM Output:
  content:    The file `greet.py` with the `greet(name)` function has been created.
  tool_call

s02 >>  q


## What Changed From S01

| Component      | Before (S01)           | After (S02)                          |
|----------------|------------------------|--------------------------------------|
| Tools          | 1 (powershell only)    | 4 (powershell, read, write, edit)    |
| Dispatch       | Hardcoded handler      | `TOOL_HANDLERS` dict lookup          |
| Path safety    | None                   | `safe_path()` sandbox                |
| Agent loop     | Unchanged              | **Unchanged**                        |

> **Key insight**: Add a tool = add a handler + add a schema entry. The loop never changes.

---

**Next: Session 03 — TodoWrite** → an agent without a plan drifts. Adding a planning layer with stateful task tracking.

# S03: TodoWrite

`s01 > s02 > [ s03 ] > s04 > s05 > s06 > s07 > s08 > s09 > s10 > s11`

> *"An agent without a plan drifts"* — list the steps first, then execute.
>
> **Harness layer**: Planning — keeping the model on course without scripting the route.

## Problem

On multi-step tasks, the model loses track. It repeats work, skips steps, or wanders off. Long conversations make this worse — the system prompt fades as tool results fill the context. A 10-step refactoring might complete steps 1–3, then the model starts improvising because it forgot steps 4–10.

## Solution

```
+--------+      +-------+      +---------+
|  User  | ---> |  LLM  | ---> | Tools   |
| prompt |      |       |      | + todo  |
+--------+      +---+---+      +----+----+
                    ^                |
                    |   tool_result  |
                    +----------------+
                          |
              +-----------+-----------+
              | TodoManager state     |
              | [ ] task A            |
              | [>] task B  <- doing  |
              | [x] task C            |
              +-----------------------+
                          |
              if rounds_since_todo >= 3:
                inject <reminder>
```

Two mechanisms:
1. **TodoManager** — structured state the LLM writes to. Only one `in_progress` at a time (forces sequential focus).
2. **Nag reminder** — if the model goes 3+ rounds without calling `todo`, inject a nudge into the conversation.

### Step 1: TodoManager

A simple class that stores todo items with statuses: `pending`, `in_progress`, `completed`. Key constraint: **only one task can be `in_progress` at a time** — this forces the model to focus.

In [109]:
class TodoManager:
    def __init__(self):
        self.items = []

    def update(self, items: list) -> str:
        """Validate and update the todo list."""
        if len(items) > 20:
            raise ValueError("Max 20 todos allowed")
        validated = []
        in_progress_count = 0
        for i, item in enumerate(items):
            text = str(item.get("text", "")).strip()
            status = str(item.get("status", "pending")).lower()
            item_id = str(item.get("id", str(i + 1)))
            if not text:
                raise ValueError(f"Item {item_id}: text required")
            if status not in ("pending", "in_progress", "completed"):
                raise ValueError(f"Item {item_id}: invalid status '{status}'")
            if status == "in_progress":
                in_progress_count += 1
            validated.append({"id": item_id, "text": text, "status": status})
        if in_progress_count > 1:
            raise ValueError("Only one task can be in_progress at a time")
        self.items = validated
        return self.render()

    def render(self) -> str:
        """Display the current todo state."""
        if not self.items:
            return "No todos."
        lines = []
        for item in self.items:
            marker = {"pending": "[ ]", "in_progress": "[>]", "completed": "[x]"}[item["status"]]
            lines.append(f"{marker} #{item['id']}: {item['text']}")
        done = sum(1 for t in self.items if t["status"] == "completed")
        lines.append(f"\n({done}/{len(self.items)} completed)")
        return "\n".join(lines)


TODO = TodoManager()

# Quick test
result = TODO.update([
    {"id": "1", "text": "Read the file", "status": "completed"},
    {"id": "2", "text": "Add type hints", "status": "in_progress"},
    {"id": "3", "text": "Write tests", "status": "pending"},
])
print(result)

[x] #1: Read the file
[>] #2: Add type hints
[ ] #3: Write tests

(1/3 completed)


### Step 2: Register the `todo` Tool

The `todo` tool goes into the dispatch map like any other tool. Same pattern as S02 — add a handler + add a schema.

In [110]:
# Reset for a fresh session
TODO = TodoManager()

# Update SYSTEM to mention todo planning
SYSTEM = f"""You are a coding agent at {WORKDIR}.  If you find we need to call tool, never output raw JSON — always call tools via function calling.
Use the todo tool to plan multi-step tasks. Mark in_progress before starting, completed when done.
Prefer tools over prose."""

# Add todo to the tool list
TOOLS = [
    {"type": "function", "function": {
        "name": "powershell",
        "description": "Run a PowerShell command.",
        "parameters": {"type": "object", "properties": {"command": {"type": "string"}}, "required": ["command"]},
    }},
    {"type": "function", "function": {
        "name": "read_file",
        "description": "Read file contents. Use 'limit' to cap line count.",
        "parameters": {"type": "object", "properties": {"path": {"type": "string"}, "limit": {"type": "integer"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "write_file",
        "description": "Write content to a file (creates parent dirs).",
        "parameters": {"type": "object", "properties": {"path": {"type": "string"}, "content": {"type": "string"}}, "required": ["path", "content"]},
    }},
    {"type": "function", "function": {
        "name": "edit_file",
        "description": "Replace exact text in a file (first occurrence).",
        "parameters": {"type": "object", "properties": {"path": {"type": "string"}, "old_text": {"type": "string"}, "new_text": {"type": "string"}}, "required": ["path", "old_text", "new_text"]},
    }},
    {"type": "function", "function": {
        "name": "todo",
        "description": "Update task list. Track progress on multi-step tasks. Set status to in_progress before starting, completed when done.",
        "parameters": {
            "type": "object",
            "properties": {
                "items": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "id": {"type": "string"},
                            "text": {"type": "string"},
                            "status": {"type": "string", "enum": ["pending", "in_progress", "completed"]},
                        },
                        "required": ["id", "text", "status"],
                    },
                },
            },
            "required": ["items"],
        },
    }},
]

# Add todo handler to the dispatch map
TOOL_HANDLERS = {
    "powershell": run_powershell,
    "read_file":  lambda **kw: run_read(kw["path"], kw.get("limit")),
    "write_file": lambda **kw: run_write(kw["path"], kw["content"]),
    "edit_file":  lambda **kw: run_edit(kw["path"], kw["old_text"], kw["new_text"]),
    "todo":       lambda **kw: TODO.update(kw["items"]),
}

print(f"Registered {len(TOOL_HANDLERS)} tools: {list(TOOL_HANDLERS.keys())}")

Registered 5 tools: ['powershell', 'read_file', 'write_file', 'edit_file', 'todo']


### Step 3: Agent Loop with Nag Reminder

The loop is almost the same as S02, with one addition: a **nag reminder**.

If the model goes 3+ rounds without calling `todo`, we inject `<reminder>Update your todos.</reminder>` into the next message. This creates accountability — the model can’t just forget its plan.

For Azure OpenAI, we inject the reminder as an extra `user` message (since tool results are separate messages, not a list).

In [111]:
def agent_loop(messages: list):
    """Agent loop with todo nag reminder."""
    turn = 0
    rounds_since_todo = 0
    while True:
        turn += 1
        all_messages = [{"role": "system", "content": SYSTEM}] + messages

        # Inject nag reminder if model forgot to update todos
        if rounds_since_todo >= 3:
            all_messages.append({"role": "user", "content": "<reminder>Update your todos.</reminder>"})
            print(f"  [nag] Injected todo reminder (no todo call for {rounds_since_todo} rounds)")

        # Show turn info
        print(f"\n{'='*60}")
        print(f"Turn {turn} - LLM Input ({len(all_messages)} messages):")
        print(f"{'='*60}")
        for j, m in enumerate(all_messages):
            role = m["role"]
            if role == "system":
                print(f"  [{j}] system: {m['content'][:80]}...")
            elif role == "user":
                content = m.get("content", "")
                if isinstance(content, str):
                    print(f"  [{j}] user: {content[:100]}")
                else:
                    print(f"  [{j}] user: (tool results)")
            elif role == "assistant":
                tc = m.get("tool_calls")
                if tc:
                    print(f"  [{j}] assistant: (called {len(tc)} tool(s))")
                else:
                    print(f"  [{j}] assistant: {str(m.get('content', ''))[:100]}")
            elif role == "tool":
                print(f"  [{j}] tool: {m['content'][:80]}")

        # Call the LLM
        response = client.chat.completions.create(
            model=MODEL,
            messages=all_messages,
            tools=TOOLS,
        )
        msg = response.choices[0].message

        # Show LLM output
        print(f"\nTurn {turn} - LLM Output:")
        print(f"  content:    {msg.content}")
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  tool_call:  {tc.function.name}({tc.function.arguments[:100]})")
        else:
            print(f"  tool_calls: None (done!)")

        # Append assistant turn
        assistant_msg = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(assistant_msg)

        # If no tool calls, we're done
        if not msg.tool_calls:
            print(f"\n>>> Agent finished in {turn} turn(s)")
            print(f"\n--- Final Todo State ---")
            print(TODO.render())
            return

        # Execute each tool call
        used_todo = False
        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            handler = TOOL_HANDLERS.get(name)
            try:
                output = handler(**args) if handler else f"Unknown tool: {name}"
            except Exception as e:
                output = f"Error: {e}"
            print(f"\n  [exec] {name}: {json.dumps(args)[:100]}")
            print(f"  [result] {str(output)[:300]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(output),
            })
            if name == "todo":
                used_todo = True

        # Track rounds since last todo call
        rounds_since_todo = 0 if used_todo else rounds_since_todo + 1

## Try It!

Give the agent a multi-step task and watch it plan with `todo`:
1. `Create a Python package with __init__.py, utils.py, and tests/test_utils.py`
2. `Refactor greet.py: add type hints, docstrings, and a main guard, make a plan first`
4. `Review all Python files in this directory and fix any style issues`

Watch for:
- The model calling `todo` to create a plan before starting
- Status transitions: `pending` → `in_progress` → `completed`
- The nag reminder if the model forgets to update its todos

In [112]:
# Interactive REPL — type your prompts, press Enter. Type "q" to quit.
TODO = TodoManager()  # Reset todos for fresh session
history = []
while True:
    try:
        query = input("s03 >> ")
    except (EOFError, KeyboardInterrupt):
        break
    if query.strip().lower() in ("q", "exit", ""):
        break
    history.append({"role": "user", "content": query})
    agent_loop(history)
    print()

s03 >>  Refactor greet.py: add type hints, docstrings, and a main guard



Turn 1 - LLM Input (2 messages):
  [0] system: You are a coding agent at C:\Users\concao\code\agent101.  If you find we need to...
  [1] user: Refactor greet.py: add type hints, docstrings, and a main guard

Turn 1 - LLM Output:
  content:    None
  tool_call:  todo({"items":[{"id":"refactor_greet","text":"Refactor greet.py: add type hints, docstrings, and a main g)

  [exec] todo: {"items": [{"id": "refactor_greet", "text": "Refactor greet.py: add type hints, docstrings, and a ma
  [result] [>] #refactor_greet: Refactor greet.py: add type hints, docstrings, and a main guard

(0/1 completed)

Turn 2 - LLM Input (4 messages):
  [0] system: You are a coding agent at C:\Users\concao\code\agent101.  If you find we need to...
  [1] user: Refactor greet.py: add type hints, docstrings, and a main guard
  [2] assistant: (called 1 tool(s))
  [3] tool: [>] #refactor_greet: Refactor greet.py: add type hints, docstrings, and a main g

Turn 2 - LLM Output:
  content:    None
  tool_call:  read_fil

s03 >>  q


## What Changed From S02

| Component      | Before (S02)           | After (S03)                          |
|----------------|------------------------|--------------------------------------|
| Tools          | 4                      | 5 (+todo)                            |
| Planning       | None                   | TodoManager with statuses            |
| Nag injection  | None                   | `<reminder>` after 3 rounds          |
| Agent loop     | Simple dispatch        | + rounds_since_todo counter          |

> **Key insight**: The model can track its own progress — and you can see it. The "one in_progress" constraint forces sequential focus. The nag creates accountability.

---

**Next: Session 04 — Subagent** → break big tasks down; each subtask gets a clean context.

# S04: Subagents

`s01 > s02 > s03 > [ s04 ] > s05 > s06 > s07 > s08 > s09 > s10 > s11`

> *"Break big tasks down; each subtask gets a clean context"*
>
> **Harness layer**: Context isolation — child agents run in fresh message lists.

## Problem

Complex tasks pollute the main context. A refactoring that requires reading 10 files, planning, and writing code fills the message list with tool results. By turn 30, the model is confused by its own history. Subtasks deserve a **clean slate** — and the parent only needs the final answer.

## Solution

```
+--------+      +-------+      +---------+
|  User  | ---> |  LLM  | ---> |  task   |---> run_subagent(prompt)
| prompt |      |       |      |  tool   |         |
+--------+      +---+---+      +----+----+    +----v-----+
                    ^                |         | Child LLM|
                    |   final text   |         | fresh [] |
                    +----------------+         | no task  |
                                               +----------+
```

The parent gets a `task` tool. The child gets all base tools **except** `task` (prevents recursion). Only the child's final text bubbles up.

### Step 1: Subagent System Prompt & Child Tools

The child agent gets a focused system prompt and the base tools (no `task` to prevent infinite spawning).

In [113]:
SUBAGENT_SYSTEM = (
    f"You are a sub-agent working in {WORKDIR}. "
    "Solve the given task using your tools. Be concise and thorough. "
    "When done, reply with a text summary of what you did."
)

# Child gets all base tools except 'task' (no recursion)
CHILD_TOOLS = [
    {"type": "function", "function": {
        "name": "powershell",
        "description": "Run a PowerShell command.",
        "parameters": {"type": "object", "properties": {"command": {"type": "string"}}, "required": ["command"]},
    }},
    {"type": "function", "function": {
        "name": "read_file",
        "description": "Read file contents. Use 'limit' to cap line count.",
        "parameters": {"type": "object", "properties": {"path": {"type": "string"}, "limit": {"type": "integer"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "write_file",
        "description": "Write content to a file (creates parent dirs).",
        "parameters": {"type": "object", "properties": {"path": {"type": "string"}, "content": {"type": "string"}}, "required": ["path", "content"]},
    }},
    {"type": "function", "function": {
        "name": "edit_file",
        "description": "Replace exact text in a file (first occurrence).",
        "parameters": {"type": "object", "properties": {"path": {"type": "string"}, "old_text": {"type": "string"}, "new_text": {"type": "string"}}, "required": ["path", "old_text", "new_text"]},
    }},
    {"type": "function", "function": {
        "name": "todo",
        "description": "Update task list. Track progress on multi-step tasks.",
        "parameters": {
            "type": "object",
            "properties": {
                "items": {"type": "array", "items": {"type": "object", "properties": {
                    "id": {"type": "string"}, "text": {"type": "string"},
                    "status": {"type": "string", "enum": ["pending", "in_progress", "completed"]}
                }, "required": ["id", "text", "status"]}},
            },
            "required": ["items"],
        },
    }},
]

# Child tool handlers (same as parent minus 'task')
CHILD_HANDLERS = {
    "powershell": run_powershell,
    "read_file":  lambda **kw: run_read(kw["path"], kw.get("limit")),
    "write_file": lambda **kw: run_write(kw["path"], kw["content"]),
    "edit_file":  lambda **kw: run_edit(kw["path"], kw["old_text"], kw["new_text"]),
    "todo":       lambda **kw: TODO.update(kw["items"]),
}

print(f"Child tools: {[t['function']['name'] for t in CHILD_TOOLS]}")

Child tools: ['powershell', 'read_file', 'write_file', 'edit_file', 'todo']


### Step 2: The Subagent Runner

`run_subagent(prompt)` spawns a child with **fresh messages**. It runs its own tool loop (max 30 iterations) and returns only the final text to the parent.

In [114]:
def run_subagent(prompt: str) -> str:
    '''Spawn a child agent with fresh context. Returns only final text.'''
    print(f"\n  [subagent] Starting child for: {prompt[:80]}...")
    sub_messages = [{"role": "user", "content": prompt}]
    for i in range(30):
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "system", "content": SUBAGENT_SYSTEM}] + sub_messages,
            tools=CHILD_TOOLS,
        )
        msg = response.choices[0].message
        assistant_msg = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        sub_messages.append(assistant_msg)
        if not msg.tool_calls:
            print(f"  [subagent] Done in {i+1} turn(s)")
            break
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            handler = CHILD_HANDLERS.get(name)
            try:
                output = handler(**args) if handler else f"Unknown tool: {name}"
            except Exception as e:
                output = f"Error: {e}"
            print(f"    [child-exec] {name}: {json.dumps(args)[:80]}")
            sub_messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(output)[:50000]})
    return msg.content or "(no summary)"

print("run_subagent() defined")

run_subagent() defined


### Step 3: Parent Tools & Dispatch Map

The parent gets all child tools **plus** the `task` tool. When the LLM calls `task`, it dispatches to `run_subagent()`.

In [115]:
# Parent tools = child tools + task
TASK_TOOL = {"type": "function", "function": {
    "name": "task",
    "description": "Delegate a subtask to a child agent with clean context. Returns the child's summary.",
    "parameters": {
        "type": "object",
        "properties": {
            "prompt": {"type": "string", "description": "The task for the child agent to solve."},
        },
        "required": ["prompt"],
    },
}}

TOOLS = CHILD_TOOLS + [TASK_TOOL]

TOOL_HANDLERS = {
    **CHILD_HANDLERS,
    "task": lambda **kw: run_subagent(kw["prompt"]),
}

SYSTEM = (
    f"You are a coding agent at {WORKDIR}. "
    "Use the 'task' tool to delegate complex subtasks to child agents. "
    "Each child gets a clean context. Use todo to plan, then delegate. "
    "Prefer tools over prose."
)

TODO = TodoManager()
print(f"Parent tools: {[t['function']['name'] for t in TOOLS]}")

Parent tools: ['powershell', 'read_file', 'write_file', 'edit_file', 'todo', 'task']


### Step 4: Agent Loop (with subagent support)

The loop is the same structure — the `task` tool handler does all the subagent work. The loop just dispatches as usual.

In [116]:
def agent_loop(messages: list):
    '''Agent loop with subagent support via task tool.'''
    turn = 0
    rounds_since_todo = 0
    while True:
        turn += 1
        all_messages = [{"role": "system", "content": SYSTEM}] + messages

        # Nag reminder from S03
        if rounds_since_todo >= 3:
            all_messages.append({"role": "user", "content": "<reminder>Update your todos.</reminder>"})
            print(f"  [nag] Injected todo reminder")

        # Show turn info
        print(f"\n{'='*60}")
        print(f"Turn {turn} - LLM Input ({len(all_messages)} messages):")
        print(f"{'='*60}")
        for j, m in enumerate(all_messages):
            role = m["role"]
            if role == "system":
                print(f"  [{j}] system: {m['content'][:80]}...")
            elif role == "user":
                content = m.get("content", "")
                if isinstance(content, str):
                    print(f"  [{j}] user: {content[:100]}")
                else:
                    print(f"  [{j}] user: (tool results)")
            elif role == "assistant":
                tc = m.get("tool_calls")
                if tc:
                    print(f"  [{j}] assistant: (called {len(tc)} tool(s))")
                else:
                    print(f"  [{j}] assistant: {str(m.get('content', ''))[:100]}")
            elif role == "tool":
                print(f"  [{j}] tool: {m['content'][:80]}")

        # Call the LLM
        response = client.chat.completions.create(
            model=MODEL,
            messages=all_messages,
            tools=TOOLS,
        )
        msg = response.choices[0].message

        # Show LLM output
        print(f"\nTurn {turn} - LLM Output:")
        print(f"  content:    {msg.content}")
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  tool_call:  {tc.function.name}({tc.function.arguments[:100]})")
        else:
            print(f"  tool_calls: None (done!)")

        # Append assistant turn
        assistant_msg = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(assistant_msg)

        # If no tool calls, we're done
        if not msg.tool_calls:
            print(f"\n>>> Agent finished in {turn} turn(s)")
            return

        # Execute each tool call
        used_todo = False
        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            handler = TOOL_HANDLERS.get(name)
            try:
                output = handler(**args) if handler else f"Unknown tool: {name}"
            except Exception as e:
                output = f"Error: {e}"
            print(f"\n  [exec] {name}: {json.dumps(args)[:100]}")
            print(f"  [result] {str(output)[:300]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(output),
            })
            if name == "todo":
                used_todo = True

        rounds_since_todo = 0 if used_todo else rounds_since_todo + 1

## Try It!

Give the agent a task that benefits from delegation:
1. `Create 3 Python files: utils.py, math_helpers.py, and string_helpers.py with useful functions in each, please use sub agent for each`

Watch the agent use `task` to spawn child agents with clean contexts.

In [117]:
# Interactive REPL
TODO = TodoManager()
history = []
while True:
    try:
        query = input("s04 >> ")
    except (EOFError, KeyboardInterrupt):
        break
    if query.strip().lower() in ("q", "exit", ""):
        break
    history.append({"role": "user", "content": query})
    agent_loop(history)
    print()

s04 >>  Create 3 Python files: utils.py, math_helpers.py, and string_helpers.py with useful functions in each, please use sub agent for each



Turn 1 - LLM Input (2 messages):
  [0] system: You are a coding agent at C:\Users\concao\code\agent101. Use the 'task' tool to ...
  [1] user: Create 3 Python files: utils.py, math_helpers.py, and string_helpers.py with useful functions in eac

Turn 1 - LLM Output:
  content:    None
  tool_call:  todo({"items":[{"id":"utils","text":"Create utils.py with general utility functions","status":"pending"},)

  [exec] todo: {"items": [{"id": "utils", "text": "Create utils.py with general utility functions", "status": "pend
  [result] [ ] #utils: Create utils.py with general utility functions
[ ] #math_helpers: Create math_helpers.py with mathematical helper functions
[ ] #string_helpers: Create string_helpers.py with string manipulation helper functions

(0/3 completed)

Turn 2 - LLM Input (4 messages):
  [0] system: You are a coding agent at C:\Users\concao\code\agent101. Use the 'task' tool to ...
  [1] user: Create 3 Python files: utils.py, math_helpers.py, and string_helpers.py with use

s04 >>  q


## What Changed From S03

| Component      | Before (S03)           | After (S04)                          |
|----------------|------------------------|--------------------------------------|
| Tools          | 5                      | 6 (+task)                            |
| Subagent       | None                   | `run_subagent()` with fresh context  |
| Child isolation | N/A                   | CHILD_TOOLS (no task = no recursion) |
| Agent loop     | + todo nag             | + subagent dispatch (via handler)    |

> **Key insight**: The child agent gets a clean `messages=[]`. Only its final text returns to the parent. This keeps the parent's context clean.

---

**Next: Session 05 — Skill Loading** → load knowledge when you need it, not upfront.

# S05: Skill Loading

`s01 > s02 > s03 > s04 > [ s05 ] > s06 > s07 > s08 > s09 > s10 > s11`

> *"Load knowledge when you need it, not upfront"*
>
> **Harness layer**: On-demand knowledge — two-layer injection keeps the system prompt small.

## Problem

Stuffing all domain knowledge into the system prompt wastes tokens and dilutes focus. A 5000-token system prompt about code review, testing patterns, and API conventions means the model always pays the cost even when only doing a simple file rename. We need knowledge **on demand**.

## Solution

```
+------------------+
| System Prompt    |       skills/
|  ...             |       +-- code-review/
|  Available:      |       |   +-- SKILL.md   (~2000 tokens)
|  - code-review   | <--+  +-- testing/
|    (~100 tokens)  |    |     +-- SKILL.md
+------------------+    |
                        |
  load_skill("code-review")
     |
     +-> injects full body into context
```

**Layer 1**: Skill names + short descriptions in system prompt (~100 tokens each).
**Layer 2**: Full skill body loaded via `load_skill` tool (~2000 tokens, on demand).

### Step 1: Create a Sample Skill

Skills live in `skills/<name>/SKILL.md` with YAML frontmatter. Let's create one for testing.

In [118]:
import yaml

# Create skills directory and a sample skill
skill_dir = WORKDIR / "skills" / "code-review"
skill_dir.mkdir(parents=True, exist_ok=True)

skill_content = (
    "---\n"
    "name: code-review\n"
    'description: "Guidelines for reviewing Python code: style, bugs, security."\n'
    "---\n"
    "\n"
    "# Code Review Skill\n"
    "\n"
    "When reviewing Python code, check for:\n"
    "\n"
    "## Style\n"
    "- PEP 8 compliance (naming, spacing, line length)\n"
    "- Consistent use of type hints\n"
    "- Docstrings on public functions\n"
    "\n"
    "## Bugs\n"
    "- Off-by-one errors in loops\n"
    "- Unclosed file handles (use context managers)\n"
    "- Mutable default arguments\n"
    "\n"
    "## Security\n"
    "- No hardcoded secrets or API keys\n"
    "- Input validation on user-supplied data\n"
    "- Safe use of subprocess (no shell=True with user input)\n"
)

(skill_dir / "SKILL.md").write_text(skill_content, encoding="utf-8")
print(f"Created {skill_dir / 'SKILL.md'}")

Created C:\Users\concao\code\agent101\skills\code-review\SKILL.md


### Step 2: SkillLoader Class

Scans `skills/*/SKILL.md`, parses YAML frontmatter. `get_descriptions()` returns a short list for the system prompt. `get_content(name)` returns the full skill body wrapped in `<skill>` tags.

In [60]:
class SkillLoader:
    def __init__(self, skills_dir):
        self.skills_dir = Path(skills_dir)
        self.skills = {}
        self._scan()

    def _scan(self):
        '''Scan skills directory for SKILL.md files with YAML frontmatter.'''
        if not self.skills_dir.exists():
            return
        for skill_file in self.skills_dir.glob("*/SKILL.md"):
            text = skill_file.read_text(encoding="utf-8")
            # Parse YAML frontmatter between --- markers
            if text.startswith("---"):
                parts = text.split("---", 2)
                if len(parts) >= 3:
                    meta = yaml.safe_load(parts[1])
                    body = parts[2].strip()
                    name = meta.get("name", skill_file.parent.name)
                    self.skills[name] = {
                        "description": meta.get("description", ""),
                        "body": body,
                    }

    def get_descriptions(self) -> str:
        '''Short descriptions for the system prompt (~100 tokens each).'''
        if not self.skills:
            return "No skills loaded."
        lines = ["Available skills (use load_skill to get full details):"]
        for name, info in self.skills.items():
            lines.append(f"  - {name}: {info['description']}")
        return "\n".join(lines)

    def get_content(self, name: str) -> str:
        '''Full skill body wrapped in <skill> tags.'''
        skill = self.skills.get(name)
        if not skill:
            available = ", ".join(self.skills.keys()) or "none"
            return f"Skill '{name}' not found. Available: {available}"
        return f"<skill name='{name}'>\n{skill['body']}\n</skill>"


SKILLS = SkillLoader(WORKDIR / "skills")
print("Loaded skills:")
print(SKILLS.get_descriptions())

Loaded skills:
Available skills (use load_skill to get full details):
  - code-review: Guidelines for reviewing Python code: style, bugs, security.


### Step 3: Register `load_skill` Tool & Update System Prompt

Skill descriptions go into the system prompt (Layer 1). The `load_skill` tool provides full content on demand (Layer 2).

In [119]:
SYSTEM = (
    f"You are a coding agent at {WORKDIR}.\n"
    f"Use todo to plan multi-step tasks. Prefer tools over prose.\n\n"
    f"{SKILLS.get_descriptions()}"
)

TOOLS = CHILD_TOOLS + [
    TASK_TOOL,
    {"type": "function", "function": {
        "name": "load_skill",
        "description": "Load full content of a skill by name. Use when you need domain-specific guidance.",
        "parameters": {
            "type": "object",
            "properties": {"name": {"type": "string", "description": "Skill name to load."}},
            "required": ["name"],
        },
    }},
]

TOOL_HANDLERS = {
    **CHILD_HANDLERS,
    "task":       lambda **kw: run_subagent(kw["prompt"]),
    "load_skill": lambda **kw: SKILLS.get_content(kw["name"]),
}

TODO = TodoManager()
print(f"Tools: {[t['function']['name'] for t in TOOLS]}")
print(f"\nSystem prompt ({len(SYSTEM)} chars):")
print(SYSTEM[:300])

Tools: ['powershell', 'read_file', 'write_file', 'edit_file', 'todo', 'task', 'load_skill']

System prompt (250 chars):
You are a coding agent at C:\Users\concao\code\agent101.
Use todo to plan multi-step tasks. Prefer tools over prose.

Available skills (use load_skill to get full details):
  - code-review: Guidelines for reviewing Python code: style, bugs, security.


### Step 4: Agent Loop

Same loop as S04 — the `load_skill` handler does the work. No loop changes needed for skill loading.

In [120]:
def agent_loop(messages: list):
    '''Agent loop with skill loading + subagent support.'''
    turn = 0
    rounds_since_todo = 0
    while True:
        turn += 1
        all_messages = [{"role": "system", "content": SYSTEM}] + messages

        if rounds_since_todo >= 3:
            all_messages.append({"role": "user", "content": "<reminder>Update your todos.</reminder>"})
            print(f"  [nag] Injected todo reminder")

        print(f"\n{'='*60}")
        print(f"Turn {turn} - LLM Input ({len(all_messages)} messages):")
        print(f"{'='*60}")
        for j, m in enumerate(all_messages):
            role = m["role"]
            if role == "system":
                print(f"  [{j}] system: {m['content'][:80]}...")
            elif role == "user":
                content = m.get("content", "")
                if isinstance(content, str):
                    print(f"  [{j}] user: {content[:100]}")
                else:
                    print(f"  [{j}] user: (tool results)")
            elif role == "assistant":
                tc = m.get("tool_calls")
                if tc:
                    print(f"  [{j}] assistant: (called {len(tc)} tool(s))")
                else:
                    print(f"  [{j}] assistant: {str(m.get('content', ''))[:100]}")
            elif role == "tool":
                print(f"  [{j}] tool: {m['content'][:80]}")

        response = client.chat.completions.create(
            model=MODEL,
            messages=all_messages,
            tools=TOOLS,
        )
        msg = response.choices[0].message

        print(f"\nTurn {turn} - LLM Output:")
        print(f"  content:    {msg.content}")
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  tool_call:  {tc.function.name}({tc.function.arguments[:100]})")
        else:
            print(f"  tool_calls: None (done!)")

        assistant_msg = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(assistant_msg)

        if not msg.tool_calls:
            print(f"\n>>> Agent finished in {turn} turn(s)")
            return

        used_todo = False
        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            handler = TOOL_HANDLERS.get(name)
            try:
                output = handler(**args) if handler else f"Unknown tool: {name}"
            except Exception as e:
                output = f"Error: {e}"
            print(f"\n  [exec] {name}: {json.dumps(args)[:100]}")
            print(f"  [result] {str(output)[:300]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(output),
            })
            if name == "todo":
                used_todo = True

        rounds_since_todo = 0 if used_todo else rounds_since_todo + 1

## Try It!

Try a task that triggers skill loading:
1. `Review the greet.py file for code quality issues`
2. `Load the code-review skill and then check pyproject.toml`

Watch the agent load the skill on demand instead of having it in the system prompt always.

In [121]:
# Interactive REPL
TODO = TodoManager()
SKILLS = SkillLoader(WORKDIR / "skills")  # Rescan
history = []
while True:
    try:
        query = input("s05 >> ")
    except (EOFError, KeyboardInterrupt):
        break
    if query.strip().lower() in ("q", "exit", ""):
        break
    history.append({"role": "user", "content": query})
    agent_loop(history)
    print()

s05 >>  Review the greet.py file for code quality issues



Turn 1 - LLM Input (2 messages):
  [0] system: You are a coding agent at C:\Users\concao\code\agent101.
Use todo to plan multi-...
  [1] user: Review the greet.py file for code quality issues

Turn 1 - LLM Output:
  content:    None
  tool_call:  read_file({"path":"C:\\Users\\concao\\code\\agent101\\greet.py","limit":200})

  [exec] read_file: {"path": "C:\\Users\\concao\\code\\agent101\\greet.py", "limit": 200}
  [result] """greet.py - A simple greeting script.

This module defines a function to print a personalized greeting.
"""

def greet(name: str) -> None:
    """Print a personalized greeting.

    Args:
        name (str): The name of the person to greet.
    """
    print(f"Hello, {name}!")


def main() -> None

Turn 2 - LLM Input (4 messages):
  [0] system: You are a coding agent at C:\Users\concao\code\agent101.
Use todo to plan multi-...
  [1] user: Review the greet.py file for code quality issues
  [2] assistant: (called 1 tool(s))
  [3] tool: """greet.py - A simple greetin

s05 >>  q


## What Changed From S04

| Component      | Before (S04)           | After (S05)                              |
|----------------|------------------------|------------------------------------------|
| Tools          | 6                      | 7 (+load_skill)                          |
| Knowledge      | All in system prompt   | Layer 1: names, Layer 2: on-demand body  |
| SkillLoader    | None                   | Scans skills/*/SKILL.md, serves content  |
| System prompt  | Static                 | Dynamic with skill descriptions          |

> **Key insight**: ~100 tokens per skill in the system prompt vs ~2000 tokens loaded on demand. With 20 skills, that saves ~38,000 tokens per turn.

---

**Next: Session 06 — Context Compact** → context will fill up; you need a way to make room.

# S06: Context Compact

`s01 > s02 > s03 > s04 > s05 > [ s06 ] > s07 > s08 > s09 > s10 > s11`

> *"Context will fill up; you need a way to make room"*
>
> **Harness layer**: Compression for infinite sessions — three layers of context management.

## Problem

Every tool call adds to the message list. After 20+ turns, the context is full of stale tool results — file contents read 15 turns ago, shell output from completed tasks. The model pays attention to all of it, wasting tokens and diluting focus. Eventually you hit the context limit and the session crashes.

## Solution

```
Three compression layers:

Layer 1 -- micro_compact (every turn)
  Replace tool results >3 turns old with placeholders
  Cost: 0 LLM calls

Layer 2 -- auto_compact (when tokens > threshold)
  Save transcript to disk, LLM summarizes, replace messages
  Cost: 1 LLM call

Layer 3 -- compact tool (model decides)
  Model explicitly calls compact when it feels context is heavy
  Cost: 1 LLM call
```

Each layer is progressively more aggressive. Layer 1 runs silently every turn. Layer 2 kicks in automatically. Layer 3 gives the model agency.

### Step 1: Token Estimation

A quick-and-dirty estimate: `len(json.dumps(messages)) / 4`. Not accurate, but good enough for threshold checks.

In [64]:
import time

# Directory for saving transcripts before compaction
TRANSCRIPT_DIR = WORKDIR / ".transcripts"
TRANSCRIPT_DIR.mkdir(exist_ok=True)

TOKEN_THRESHOLD = 40000  # Auto-compact when estimated tokens exceed this

def estimate_tokens(messages: list) -> int:
    '''Rough token estimate: chars / 4.'''
    return len(json.dumps(messages, default=str)) // 4

# Test
sample = [{"role": "user", "content": "hello " * 1000}]
print(f"Sample message tokens (est): {estimate_tokens(sample)}")

Sample message tokens (est): 1508


### Step 2: Micro-Compact (Layer 1)

Every turn, replace tool result content older than 3 turns with a short placeholder. This is free (no LLM calls) and keeps the context from growing linearly.

In [65]:
def micro_compact(messages: list) -> list:
    '''Replace old tool results (>3 turns old) with placeholders.'''
    if len(messages) <= 6:
        return messages
    cutoff = len(messages) - 6  # Keep last 3 pairs (assistant + tool)
    result = []
    for i, msg in enumerate(messages):
        if i < cutoff and msg.get("role") == "tool":
            # Find which tool was called by looking at preceding assistant msg
            tool_name = "tool"
            for j in range(i - 1, -1, -1):
                tc_list = messages[j].get("tool_calls", [])
                for tc in tc_list:
                    if tc.get("id") == msg.get("tool_call_id"):
                        tool_name = tc["function"]["name"]
                        break
            result.append({
                "role": "tool",
                "tool_call_id": msg.get("tool_call_id", ""),
                "content": f"[Previous: used {tool_name}]",
            })
        else:
            result.append(msg)
    return result

print("micro_compact() defined")

micro_compact() defined


### Step 3: Auto-Compact (Layer 2) & Compact Tool (Layer 3)

When tokens exceed the threshold, save the full transcript to disk, ask the LLM to summarize, and replace the entire message list with the summary. The `compact` tool lets the model trigger this explicitly.

In [66]:
def auto_compact(messages: list) -> list:
    '''Save transcript, LLM summarizes, return compressed messages.'''
    # Save transcript to disk
    transcript_path = TRANSCRIPT_DIR / f"transcript_{int(time.time())}.jsonl"
    with open(transcript_path, "w", encoding="utf-8") as f:
        for msg in messages:
            f.write(json.dumps(msg, default=str) + "\n")
    print(f"  [compact] Saved transcript to {transcript_path.name}")

    # LLM summarizes
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                "Summarize this conversation for continuity. "
                "Focus on what was accomplished and what's pending."
            )},
            {"role": "user", "content": json.dumps(messages, default=str)[:80000]},
        ],
    )
    summary = response.choices[0].message.content
    print(f"  [compact] Compressed {len(messages)} messages into summary")
    return [{"role": "user", "content": f"[Compressed context]\n\n{summary}"}]


def run_compact(**kwargs) -> str:
    '''Tool handler: compact is handled in the agent loop (returns placeholder).'''
    return "Manual compression request"

print("auto_compact() and run_compact() defined")

auto_compact() and run_compact() defined


### Step 4: Updated Tools & Agent Loop

The agent loop now runs all 3 compression layers:
1. `micro_compact` every turn (Layer 1)
2. `auto_compact` when tokens > threshold (Layer 2)
3. `compact` tool call triggers Layer 2 explicitly (Layer 3)

In [122]:
TOOLS = CHILD_TOOLS + [
    TASK_TOOL,
    {"type": "function", "function": {
        "name": "load_skill",
        "description": "Load full content of a skill by name.",
        "parameters": {"type": "object", "properties": {"name": {"type": "string"}}, "required": ["name"]},
    }},
    {"type": "function", "function": {
        "name": "compact",
        "description": "Compress conversation context. Use when context feels heavy or you're doing many tool calls.",
        "parameters": {"type": "object", "properties": {}},
    }},
]

TOOL_HANDLERS = {
    **CHILD_HANDLERS,
    "task":       lambda **kw: run_subagent(kw["prompt"]),
    "load_skill": lambda **kw: SKILLS.get_content(kw["name"]),
    "compact":    run_compact,
}

SYSTEM = (
    f"You are a coding agent at {WORKDIR}.\n"
    "Use todo to plan multi-step tasks. Prefer tools over prose.\n"
    "Use compact when context feels heavy after many tool calls.\n\n"
    f"{SKILLS.get_descriptions()}"
)

TODO = TodoManager()
print(f"Tools: {[t['function']['name'] for t in TOOLS]}")

Tools: ['powershell', 'read_file', 'write_file', 'edit_file', 'todo', 'task', 'load_skill', 'compact']


In [123]:
def agent_loop(messages: list):
    '''Agent loop with 3-layer context compaction.'''
    turn = 0
    rounds_since_todo = 0
    compact_requested = False
    while True:
        turn += 1

        # Layer 1: micro-compact old tool results
        messages[:] = micro_compact(messages)

        # Layer 2: auto-compact if over token threshold
        est = estimate_tokens(messages)
        if est > TOKEN_THRESHOLD or compact_requested:
            reason = "tool request" if compact_requested else f"tokens ({est}) > {TOKEN_THRESHOLD}"
            print(f"  [compact] Triggering auto_compact: {reason}")
            messages[:] = auto_compact(messages)
            compact_requested = False

        all_messages = [{"role": "system", "content": SYSTEM}] + messages

        if rounds_since_todo >= 3:
            all_messages.append({"role": "user", "content": "<reminder>Update your todos.</reminder>"})
            print(f"  [nag] Injected todo reminder")

        # Show turn info
        print(f"\n{'='*60}")
        print(f"Turn {turn} - LLM Input ({len(all_messages)} msgs, ~{estimate_tokens(all_messages)} tokens):")
        print(f"{'='*60}")
        for j, m in enumerate(all_messages):
            role = m["role"]
            if role == "system":
                print(f"  [{j}] system: {m['content'][:80]}...")
            elif role == "user":
                content = m.get("content", "")
                if isinstance(content, str):
                    print(f"  [{j}] user: {content[:100]}")
                else:
                    print(f"  [{j}] user: (tool results)")
            elif role == "assistant":
                tc = m.get("tool_calls")
                if tc:
                    print(f"  [{j}] assistant: (called {len(tc)} tool(s))")
                else:
                    print(f"  [{j}] assistant: {str(m.get('content', ''))[:100]}")
            elif role == "tool":
                print(f"  [{j}] tool: {m['content'][:80]}")

        response = client.chat.completions.create(
            model=MODEL,
            messages=all_messages,
            tools=TOOLS,
        )
        msg = response.choices[0].message

        print(f"\nTurn {turn} - LLM Output:")
        print(f"  content:    {msg.content}")
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  tool_call:  {tc.function.name}({tc.function.arguments[:100]})")
        else:
            print(f"  tool_calls: None (done!)")

        assistant_msg = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(assistant_msg)

        if not msg.tool_calls:
            print(f"\n>>> Agent finished in {turn} turn(s)")
            return

        used_todo = False
        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            handler = TOOL_HANDLERS.get(name)
            try:
                output = handler(**args) if handler else f"Unknown tool: {name}"
            except Exception as e:
                output = f"Error: {e}"
            print(f"\n  [exec] {name}: {json.dumps(args)[:100]}")
            print(f"  [result] {str(output)[:300]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(output),
            })
            if name == "todo":
                used_todo = True
            if name == "compact":
                compact_requested = True

        rounds_since_todo = 0 if used_todo else rounds_since_todo + 1

## Try It!

Try a long session and watch compaction kick in:
1. `Read every .py file in this directory and summarize each one` (generates lots of tool results)
2. `compact` (model triggers explicit compaction)
3. `What files did you just read?` (tests that the summary preserved context)

Watch the token estimates in the turn headers.

In [124]:
# Interactive REPL
TODO = TodoManager()
history = []
while True:
    try:
        query = input("s06 >> ")
    except (EOFError, KeyboardInterrupt):
        break
    if query.strip().lower() in ("q", "exit", ""):
        break
    history.append({"role": "user", "content": query})
    agent_loop(history)
    print()

s06 >>  compact



Turn 1 - LLM Input (2 msgs, ~98 tokens):
  [0] system: You are a coding agent at C:\Users\concao\code\agent101.
Use todo to plan multi-...
  [1] user: compact

Turn 1 - LLM Output:
  content:    None
  tool_call:  compact({})

  [exec] compact: {}
  [result] Manual compression request
  [compact] Triggering auto_compact: tool request
  [compact] Saved transcript to transcript_1776408175.jsonl
  [compact] Compressed 3 messages into summary

Turn 2 - LLM Input (2 msgs, ~221 tokens):
  [0] system: You are a coding agent at C:\Users\concao\code\agent101.
Use todo to plan multi-...
  [1] user: [Compressed context]

So far, you requested a **compact (compressed) version** of a task or document

Turn 2 - LLM Output:
  content:    Understood 👍  
Please provide the **content** you’d like compressed (a document, conversation, code, or notes).  
Once you share it, I’ll create a **compact version**—summarized, shortened, or abstracted—to fit your needs.
  tool_calls: None (done!)

>>> Agent finis

s06 >>  q


## What Changed From S05

| Component      | Before (S05)           | After (S06)                              |
|----------------|------------------------|------------------------------------------|
| Tools          | 7                      | 8 (+compact)                             |
| Token tracking | None                   | `estimate_tokens()` per turn             |
| Layer 1        | None                   | `micro_compact` — free, every turn        |
| Layer 2        | None                   | `auto_compact` — 1 LLM call at threshold  |
| Layer 3        | None                   | `compact` tool — model-initiated          |
| Transcripts    | None                   | Saved to `.transcripts/` before compact  |

> **Key insight**: Compression is not optional for long sessions. Layer 1 is free and always on. Layer 2 is automatic insurance. Layer 3 gives the model agency over its own context.

---

**Next: Session 07 — Task System** → break big goals into small tasks, order them, persist to disk.

# S07: Permission System

`s01 > s02 > s03 > s04 > s05 > s06 > [ s07 ] > s08 > s09 > s10 > s11`

> *"Safety is a pipeline, not a boolean."*
>
> **Harness layer**: Safety — the pipeline between intent and execution.

Your agent from S07 is capable and long-lived. It reads files, writes code, runs PowerShell, delegates subtasks, and compresses context. But there is **no safety catch**. Every tool call the model proposes goes straight to execution. Ask it to delete a directory and it will — no questions asked.

## The Problem

Imagine your agent is helping refactor a codebase. It reads a few files, proposes edits, then decides to run `Remove-Item -Recurse -Force C:\important` to clean up. Except the model hallucinated the path.

Without a permission layer, **intent becomes execution instantly**. There is no moment where the system can say *"wait, that looks dangerous"* or where you can say *"no, don't do that."*

## The Solution: Four-Stage Permission Pipeline

Every tool call passes through four stages. First match wins.

```
tool_call from LLM
     |
     v
[1. Deny rules]     -- blocklist: always block these
     |
     v
[2. Mode check]     -- plan mode? auto mode? default?
     |
     v
[3. Allow rules]    -- allowlist: always allow these
     |
     v
[4. Ask user]       -- interactive y/n/always prompt
     |
     v
execute (or reject)
```

### Three Permission Modes

| Mode | Behavior | Use Case |
|---|---|---|
| **default** | Ask user for every unmatched tool call | Normal interactive use |
| **plan** | Block all writes, allow reads | Planning/review mode |
| **auto** | Auto-allow reads, ask for writes | Fast exploration |

### Step 1: PowerShell Security Validator

Before the pipeline runs, dangerous PowerShell patterns get flagged early.

In [125]:
import re
from fnmatch import fnmatch

class PowerShellSecurityValidator:
    VALIDATORS = [
        ('remove_recurse', r'Remove-Item.*-Recurse.*-Force'),
        ('stop_computer', r'\bStop-Computer\b'),
        ('restart_computer', r'\bRestart-Computer\b'),
        ('format_volume', r'\bFormat-Volume\b'),
        ('invoke_expression', r'\bInvoke-Expression\b'),
        ('iex_alias', r'\biex\b'),
        ('execution_policy', r'Set-ExecutionPolicy'),
    ]

    def validate(self, command: str) -> list:
        failures = []
        for name, pattern in self.VALIDATORS:
            if re.search(pattern, command, re.IGNORECASE):
                failures.append((name, pattern))
        return failures

    def is_safe(self, command: str) -> bool:
        return len(self.validate(command)) == 0

    def describe_failures(self, command: str) -> str:
        failures = self.validate(command)
        if not failures: return 'No issues detected'
        parts = [f'{name}' for name, _ in failures]
        return 'Security flags: ' + ', '.join(parts)

ps_validator = PowerShellSecurityValidator()

# Test it
print(ps_validator.describe_failures('Get-ChildItem .'))
print(ps_validator.describe_failures('Remove-Item -Recurse -Force C:\\important'))
print(ps_validator.describe_failures('Stop-Computer'))

No issues detected
Security flags: remove_recurse
Security flags: stop_computer


### Step 2: Permission Rules

Rules are checked in order — **first match wins**. Deny rules use `content` for PowerShell command patterns. Allow rules use `path` for file globs.

In [126]:
MODES = ('default', 'plan', 'auto')
READ_ONLY_TOOLS = {'read_file'}
WRITE_TOOLS = {'write_file', 'edit_file', 'powershell'}

DEFAULT_RULES = [
    # Always deny dangerous patterns
    {'tool': 'powershell', 'content': '*Remove-Item*-Recurse*-Force*', 'behavior': 'deny'},
    {'tool': 'powershell', 'content': '*Stop-Computer*', 'behavior': 'deny'},
    {'tool': 'powershell', 'content': '*Restart-Computer*', 'behavior': 'deny'},
    {'tool': 'powershell', 'content': '*Format-Volume*', 'behavior': 'deny'},
    # Always allow reading files
    {'tool': 'read_file', 'path': '*', 'behavior': 'allow'},
]
print(f'{len(DEFAULT_RULES)} default rules loaded')

5 default rules loaded


### Step 3: PermissionManager

The core class implements the four-stage pipeline. Notice: deny rules **always** run first and cannot be bypassed.

In [127]:
class PermissionManager:
    def __init__(self, mode='default', rules=None):
        if mode not in MODES:
            raise ValueError(f'Unknown mode: {mode}. Choose from {MODES}')
        self.mode = mode
        self.rules = rules or list(DEFAULT_RULES)
        self.consecutive_denials = 0
        self.max_consecutive_denials = 3

    def check(self, tool_name: str, tool_input: dict) -> dict:
        # Step 0: PowerShell security validation
        if tool_name == 'powershell':
            command = tool_input.get('command', '')
            failures = ps_validator.validate(command)
            if failures:
                severe = {'remove_recurse', 'stop_computer', 'restart_computer', 'format_volume'}
                severe_hits = [f for f in failures if f[0] in severe]
                if severe_hits:
                    return {'behavior': 'deny', 'reason': f'Security: {ps_validator.describe_failures(command)}'}
                return {'behavior': 'ask', 'reason': f'Flagged: {ps_validator.describe_failures(command)}'}

        # Step 1: Deny rules (bypass-immune)
        for rule in self.rules:
            if rule['behavior'] != 'deny': continue
            if self._matches(rule, tool_name, tool_input):
                return {'behavior': 'deny', 'reason': f'Blocked by deny rule: {rule}'}

        # Step 2: Mode-based decisions
        if self.mode == 'plan':
            if tool_name in WRITE_TOOLS:
                return {'behavior': 'deny', 'reason': 'Plan mode: writes blocked'}
            return {'behavior': 'allow', 'reason': 'Plan mode: read-only allowed'}
        if self.mode == 'auto':
            if tool_name in READ_ONLY_TOOLS:
                return {'behavior': 'allow', 'reason': 'Auto mode: read-only approved'}

        # Step 3: Allow rules
        for rule in self.rules:
            if rule['behavior'] != 'allow': continue
            if self._matches(rule, tool_name, tool_input):
                self.consecutive_denials = 0
                return {'behavior': 'allow', 'reason': f'Matched allow rule'}

        # Step 4: Ask user
        return {'behavior': 'ask', 'reason': f'No rule matched for {tool_name}'}

    def ask_user(self, tool_name: str, tool_input: dict) -> bool:
        preview = json.dumps(tool_input, ensure_ascii=False)[:200]
        print(f'\n  [Permission] {tool_name}: {preview}')
        try:
            answer = input('  Allow? (y/n/always): ').strip().lower()
        except (EOFError, KeyboardInterrupt):
            return False
        if answer == 'always':
            self.rules.append({'tool': tool_name, 'path': '*', 'behavior': 'allow'})
            print(f'  [+] Added permanent allow rule for {tool_name}')
            self.consecutive_denials = 0
            return True
        if answer in ('y', 'yes'):
            self.consecutive_denials = 0
            return True
        self.consecutive_denials += 1
        if self.consecutive_denials >= self.max_consecutive_denials:
            print(f'  [{self.consecutive_denials} consecutive denials -- consider /mode plan]')
        return False

    def _matches(self, rule: dict, tool_name: str, tool_input: dict) -> bool:
        if rule.get('tool') and rule['tool'] != '*':
            if rule['tool'] != tool_name: return False
        if 'path' in rule and rule['path'] != '*':
            path = tool_input.get('path', '')
            if not fnmatch(path, rule['path']): return False
        if 'content' in rule:
            command = tool_input.get('command', '')
            if not fnmatch(command, rule['content']): return False
        return True

print('PermissionManager class ready')

PermissionManager class ready


### Step 4: Test the Pipeline

Let's verify each stage works before integrating with the agent loop.

In [128]:
perms = PermissionManager(mode='auto')

# Test deny: dangerous command
d1 = perms.check('powershell', {'command': 'Remove-Item -Recurse -Force C:\\Users'})
print(f"Dangerous PS:  {d1['behavior']} -- {d1['reason']}")

# Test auto mode: read_file auto-allowed
d2 = perms.check('read_file', {'path': 'README.md'})
print(f"Read file:     {d2['behavior']} -- {d2['reason']}")

# Test auto mode: write_file -> ask
d3 = perms.check('write_file', {'path': 'test.txt', 'content': 'hi'})
print(f"Write file:    {d3['behavior']} -- {d3['reason']}")

# Test allow rule match: read_file always allowed
d4 = perms.check('read_file', {'path': 'any/path/here.py'})
print(f"Read any path: {d4['behavior']} -- {d4['reason']}")

# Test safe powershell: should go to ask in default, but auto mode -> ask for writes
d5 = perms.check('powershell', {'command': 'Get-ChildItem .'})
print(f"Safe PS (auto): {d5['behavior']} -- {d5['reason']}")

# Test plan mode
plan_perms = PermissionManager(mode='plan')
d6 = plan_perms.check('write_file', {'path': 'x.txt', 'content': 'hi'})
print(f"Write (plan):  {d6['behavior']} -- {d6['reason']}")
d7 = plan_perms.check('read_file', {'path': 'x.txt'})
print(f"Read (plan):   {d7['behavior']} -- {d7['reason']}")

Dangerous PS:  deny -- Security: Security flags: remove_recurse
Read file:     allow -- Auto mode: read-only approved
Write file:    ask -- No rule matched for write_file
Read any path: allow -- Auto mode: read-only approved
Safe PS (auto): ask -- No rule matched for powershell
Write (plan):  deny -- Plan mode: writes blocked
Read (plan):   allow -- Plan mode: read-only allowed


### Step 5: Permission-Aware Agent Loop

Every tool call now goes through the pipeline. Three outcomes: **denied** (with reason), **allowed** (silently), or **asked** (interactively with y/n/always).

In [129]:
TOOLS = [
    {'type': 'function', 'function': {'name': 'powershell', 'description': 'Run a PowerShell command.', 'parameters': {'type': 'object', 'properties': {'command': {'type': 'string'}}, 'required': ['command']}}},
    {'type': 'function', 'function': {'name': 'read_file', 'description': 'Read file contents.', 'parameters': {'type': 'object', 'properties': {'path': {'type': 'string'}, 'limit': {'type': 'integer'}}, 'required': ['path']}}},
    {'type': 'function', 'function': {'name': 'write_file', 'description': 'Write content to a file.', 'parameters': {'type': 'object', 'properties': {'path': {'type': 'string'}, 'content': {'type': 'string'}}, 'required': ['path', 'content']}}},
    {'type': 'function', 'function': {'name': 'edit_file', 'description': 'Replace exact text in a file.', 'parameters': {'type': 'object', 'properties': {'path': {'type': 'string'}, 'old_text': {'type': 'string'}, 'new_text': {'type': 'string'}}, 'required': ['path', 'old_text', 'new_text']}}},
]
TOOL_HANDLERS = {
    'powershell': run_powershell,
    'read_file':  lambda **kw: run_read(kw['path'], kw.get('limit')),
    'write_file': lambda **kw: run_write(kw['path'], kw['content']),
    'edit_file':  lambda **kw: run_edit(kw['path'], kw['old_text'], kw['new_text']),
}
SYSTEM = f"""You are a coding agent at {WORKDIR}.
You MUST use tools via function calling. Never output raw JSON.
The user controls permissions. Some tool calls may be denied — if so, try an alternative approach."""

def agent_loop(messages: list, perms: PermissionManager):
    turn = 0
    while True:
        turn += 1
        all_messages = [{'role': 'system', 'content': SYSTEM}] + messages
        print(f"\n{'='*60}\nTurn {turn} ({len(all_messages)} msgs) [mode: {perms.mode}]\n{'='*60}")
        for j, m in enumerate(all_messages):
            role = m['role']
            if role == 'system': print(f'  [{j}] system: ...')
            elif role == 'user': print(f"  [{j}] user: {str(m.get('content',''))[:100]}")
            elif role == 'assistant':
                if m.get('tool_calls'): print(f"  [{j}] assistant: ({len(m['tool_calls'])} tool calls)")
                else: print(f"  [{j}] assistant: {str(m.get('content',''))[:100]}")
            elif role == 'tool': print(f"  [{j}] tool: {m['content'][:80]}")
        response = client.chat.completions.create(model=MODEL, messages=all_messages, tools=TOOLS)
        msg = response.choices[0].message
        print(f"\nTurn {turn} output: content={str(msg.content)[:80]}, tools={len(msg.tool_calls) if msg.tool_calls else 0}")
        assistant_msg = {'role': 'assistant', 'content': msg.content or ''}
        if msg.tool_calls:
            assistant_msg['tool_calls'] = [{'id': tc.id, 'type': 'function', 'function': {'name': tc.function.name, 'arguments': tc.function.arguments}} for tc in msg.tool_calls]
        messages.append(assistant_msg)
        if not msg.tool_calls:
            print(f'\n>>> Finished in {turn} turn(s)')
            return
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            decision = perms.check(tc.function.name, args)
            if decision['behavior'] == 'deny':
                output = f"Permission denied: {decision['reason']}"
                print(f"  [DENIED] {tc.function.name}: {decision['reason']}")
            elif decision['behavior'] == 'ask':
                if perms.ask_user(tc.function.name, args):
                    handler = TOOL_HANDLERS.get(tc.function.name)
                    try: output = handler(**args) if handler else f'Unknown: {tc.function.name}'
                    except Exception as e: output = f'Error: {e}'
                    print(f'  [APPROVED] {tc.function.name}: {str(output)[:200]}')
                else:
                    output = f'Permission denied by user for {tc.function.name}'
                    print(f'  [USER DENIED] {tc.function.name}')
            else:  # allow
                handler = TOOL_HANDLERS.get(tc.function.name)
                try: output = handler(**args) if handler else f'Unknown: {tc.function.name}'
                except Exception as e: output = f'Error: {e}'
                print(f'  [ALLOWED] {tc.function.name}: {str(output)[:200]}')
            messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': str(output)})
print('Permission-aware agent_loop ready')

Permission-aware agent_loop ready


## Try It!

1. Start in **default** mode — every write tool asks for approval
2. Try asking the agent to `delete all files` — watch deny rules block it
3. Answer **always** to permanently allow a tool type
4. Use `/mode plan` to switch — all writes blocked, reads pass through
5. Use `/mode auto` — reads auto-approved, writes still ask
6. Use `/rules` to see the current rule set
7. Use `/mode default` to go back to asking for everything

In [130]:
# Choose starting mode
print('Permission modes: default, plan, auto')
mode_choice = input('Mode (default): ').strip().lower() or 'default'
if mode_choice not in MODES: mode_choice = 'default'
perms = PermissionManager(mode=mode_choice)
print(f'[Permission mode: {perms.mode}]\n')

history = []
while True:
    try: query = input('s07 >> ')
    except (EOFError, KeyboardInterrupt): break
    if query.strip().lower() in ('q', 'exit', ''): break
    if query.startswith('/mode'):
        parts = query.split()
        if len(parts) == 2 and parts[1] in MODES:
            perms.mode = parts[1]
            print(f'[Switched to {parts[1]} mode]')
        else:
            print(f'Usage: /mode <{"|".join(MODES)}>')
        continue
    if query.strip() == '/rules':
        for i, rule in enumerate(perms.rules):
            print(f'  {i}: {rule}')
        continue
    history.append({'role': 'user', 'content': query})
    agent_loop(history, perms)
    print()

Permission modes: default, plan, auto


Mode (default):  /plan


[Permission mode: default]



s07 >>  /plan



Turn 1 (2 msgs) [mode: default]
  [0] system: ...
  [1] user: /plan

Turn 1 output: content=Here’s the plan going forward:  

1. **Understand user goal** — Determine what y, tools=0

>>> Finished in 1 turn(s)



...

  Allow? (y/n/always):  n


  [USER DENIED] powershell

Turn 2 (6 msgs) [mode: default]
  [0] system: ...
  [1] user: /plan
  [2] assistant: Here’s the plan going forward:  

1. **Understand user goal** — Determine what you’d like me to do (
  [3] user: create a file named concao.txt and echo "concao"
  [4] assistant: (1 tool calls)
  [5] tool: Permission denied by user for powershell

Turn 2 output: content=None, tools=1

  [Permission] write_file: {"path": "C:\\Users\\concao\\code\\agent101\\concao.txt", "content": "concao"}


  Allow? (y/n/always):  y


  [APPROVED] write_file: Wrote 6 bytes to C:\Users\concao\code\agent101\concao.txt

Turn 3 (8 msgs) [mode: default]
  [0] system: ...
  [1] user: /plan
  [2] assistant: Here’s the plan going forward:  

1. **Understand user goal** — Determine what you’d like me to do (
  [3] user: create a file named concao.txt and echo "concao"
  [4] assistant: (1 tool calls)
  [5] tool: Permission denied by user for powershell
  [6] assistant: (1 tool calls)
  [7] tool: Wrote 6 bytes to C:\Users\concao\code\agent101\concao.txt

Turn 3 output: content=✅ The file **`concao.txt`** has been created at `C:\Users\concao\code\agent101` , tools=0

>>> Finished in 3 turn(s)



s07 >>  q


## What Changed From S06

| Component | Before (S06) | After (S09) |
|---|---|---|
| Safety | None (direct execution) | 4-stage permission pipeline |
| Modes | None | 3 modes: default, plan, auto |
| Rules | None | Deny/allow rules with pattern matching |
| User control | None | Interactive y/n/always approval |
| Denial tracking | None | Circuit breaker after 3 denials |
| PS validation | Basic dangerous list | Regex-based security validator |

> **Key takeaway**: The permission system is a **pipeline**, not a boolean. Each stage has a clear role: deny catches the worst, mode sets the baseline, allow handles known-safe patterns, and ask is the final safety net.

---

**What you've mastered so far:**

| Session | Concept |
|---|---|
| S01 | The Agent Loop |
| S02 | Multiple Tools |
| S03 | TodoWrite (Planning) |
| S04 | Subagents (Context Isolation) |
| S05 | Skill Loading (On-Demand Knowledge) |
| S06 | Context Compact (Compression) |
| S10 | Task System (Persistent Goals) |
| S11 | MCP & Plugins (External Integration) |
| S07 | Permission System (Safety Pipeline) |

# S08: Memory System

`s01 > s02 > s03 > s04 > s05 > s06 > s07 > [ s08 ] > s09 > s10 > s11`

> *"Memory is not a dump of everything the agent has seen — it is a small store of durable facts that should still matter next session."*
>
> **Harness layer**: Persistence — remembering across the session boundary.

Your agent from S09 is powerful, safe, and extensible. But it has **amnesia**. Every time you start a new session, the agent meets you for the first time. It doesn't remember that you prefer `pnpm` over `npm`, that you told it three times to stop modifying test snapshots, or that the legacy directory can't be deleted because deployment depends on it.

## The Problem

Without memory, a new session starts from zero. The agent keeps forgetting:
- Long-term user preferences
- Corrections you've repeated multiple times
- Project constraints not obvious from the code
- External references the project depends on

You waste time re-establishing context that should have been saved once.

## The Solution: File-Based Memory Store

A small store saves durable facts as individual markdown files with YAML frontmatter. At session start, relevant memories are loaded and injected into context.

```
conversation
   |
   | durable fact appears
   v
save_memory
   |
   v
.memory/
  ├── MEMORY.md          (index)
  ├── prefer_pnpm.md     (user pref)
  ├── no_snapshots.md    (feedback)
  └── dashboard_url.md   (reference)
   |
   v
next session loads relevant entries
```

### Four Memory Categories

| Type | Purpose | Example |
|---|---|---|
| **user** | Stable user preferences | Prefers pnpm, wants concise answers |
| **feedback** | Corrections to enforce | "Don't change test snapshots unless I ask" |
| **project** | Durable project facts not obvious from code | "Legacy dir can't be deleted — deployment depends on it" |
| **reference** | Pointers to external resources | Incident board URL, spec doc location |

### What Should NOT Go Into Memory

This boundary is the most important part — and where most beginners go wrong.

| Do NOT store | Why |
|---|---|
| File tree layout | Can be re-read from the repo |
| Function names/signatures | Code is the source of truth |
| Current task status | Belongs to task/plan, not memory |
| Temporary branch names | Gets stale quickly |
| Secrets or credentials | Security risk |

**Rule**: Only keep information that still matters across sessions and cannot be cheaply re-derived from the current workspace.

### Memory vs Neighbors

| Concept | Purpose | Lifetime |
|---|---|---|
| **Memory** | Facts that survive across sessions | Persistent |
| **Task** | What the system is trying to finish now | One task |
| **Plan** | How this session intends to proceed | One session |
| **CLAUDE.md** | Stable instructions and project rules | Persistent |

Short rule: only useful for this task → use task/plan. Useful next session too → use memory.

### Step 1: MemoryManager Class

Each memory is a markdown file with YAML frontmatter: `name`, `description`, `type`, and the body content.

In [77]:
MEMORY_DIR = WORKDIR / '.memory'
MEMORY_INDEX = MEMORY_DIR / 'MEMORY.md'
MEMORY_TYPES = ('user', 'feedback', 'project', 'reference')
MAX_INDEX_LINES = 200

class MemoryManager:
    def __init__(self, memory_dir: Path = None):
        self.memory_dir = memory_dir or MEMORY_DIR
        self.memories = {}

    def load_all(self):
        self.memories = {}
        if not self.memory_dir.exists(): return
        for md_file in sorted(self.memory_dir.glob('*.md')):
            if md_file.name == 'MEMORY.md': continue
            parsed = self._parse_frontmatter(md_file.read_text(encoding='utf-8'))
            if parsed:
                name = parsed.get('name', md_file.stem)
                self.memories[name] = {
                    'description': parsed.get('description', ''),
                    'type': parsed.get('type', 'project'),
                    'content': parsed.get('content', ''),
                    'file': md_file.name,
                }
        if self.memories:
            print(f'[Memory] Loaded {len(self.memories)} memories')

    def load_memory_prompt(self) -> str:
        if not self.memories: return ''
        sections = ['# Memories (persistent across sessions)', '']
        for mem_type in MEMORY_TYPES:
            typed = {k: v for k, v in self.memories.items() if v['type'] == mem_type}
            if not typed: continue
            sections.append(f'## [{mem_type}]')
            for name, mem in typed.items():
                sections.append(f'### {name}: {mem["description"]}')
                if mem['content'].strip():
                    sections.append(mem['content'].strip())
                sections.append('')
        return '\n'.join(sections)

    def save_memory(self, name: str, description: str, mem_type: str, content: str) -> str:
        if mem_type not in MEMORY_TYPES:
            return f'Error: type must be one of {MEMORY_TYPES}'
        import re as _re
        safe_name = _re.sub(r'[^a-zA-Z0-9_-]', '_', name.lower())
        if not safe_name: return 'Error: invalid memory name'
        self.memory_dir.mkdir(parents=True, exist_ok=True)
        frontmatter = f'---\nname: {name}\ndescription: {description}\ntype: {mem_type}\n---\n{content}\n'
        file_path = self.memory_dir / f'{safe_name}.md'
        file_path.write_text(frontmatter, encoding='utf-8')
        self.memories[name] = {'description': description, 'type': mem_type, 'content': content, 'file': f'{safe_name}.md'}
        self._rebuild_index()
        return f"Saved memory '{name}' [{mem_type}] to {file_path.relative_to(WORKDIR)}"

    def delete_memory(self, name: str) -> str:
        if name not in self.memories:
            return f"Error: memory '{name}' not found. Available: {', '.join(self.memories.keys())}"
        mem = self.memories.pop(name)
        file_path = self.memory_dir / mem['file']
        if file_path.exists(): file_path.unlink()
        self._rebuild_index()
        return f"Deleted memory '{name}'"

    def _rebuild_index(self):
        lines = ['# Memory Index', '']
        for name, mem in self.memories.items():
            lines.append(f"- {name}: {mem['description']} [{mem['type']}]")
            if len(lines) >= MAX_INDEX_LINES:
                lines.append(f'... (truncated at {MAX_INDEX_LINES} lines)')
                break
        self.memory_dir.mkdir(parents=True, exist_ok=True)
        MEMORY_INDEX.write_text('\n'.join(lines) + '\n', encoding='utf-8')

    def _parse_frontmatter(self, text: str) -> dict | None:
        import re as _re
        match = _re.match(r'^---\s*\n(.*?)\n---\s*\n(.*)', text, _re.DOTALL)
        if not match: return None
        header, body = match.group(1), match.group(2)
        result = {'content': body.strip()}
        for line in header.splitlines():
            if ':' in line:
                key, _, value = line.partition(':')
                result[key.strip()] = value.strip()
        return result

memory_mgr = MemoryManager()
memory_mgr.load_all()
print(f'MemoryManager ready. {len(memory_mgr.memories)} memories loaded.')

MemoryManager ready. 0 memories loaded.


### Step 2: Test Memory Operations

Let's save, load, and inspect memories directly before wiring them to the agent.

In [131]:
# Save some test memories
print(memory_mgr.save_memory(
    'prefer_powershell',
    'User prefers PowerShell over bash/cmd',
    'user',
    'Always use PowerShell for system commands. Do not suggest bash or cmd alternatives.'
))

print(memory_mgr.save_memory(
    'no_snapshot_changes',
    'Do not modify test snapshots unless explicitly asked',
    'feedback',
    'The user has corrected this twice. Test snapshots should only be updated when the user specifically requests it.'
))

print(memory_mgr.save_memory(
    'legacy_dir_keep',
    'Legacy directory must not be deleted',
    'project',
    'The /legacy directory looks unused but deployment scripts still reference it. Do not delete or rename.'
))

# Check the index
print('\n--- MEMORY.md ---')
print(MEMORY_INDEX.read_text(encoding='utf-8'))

Saved memory 'prefer_powershell' [user] to .memory\prefer_powershell.md
Saved memory 'no_snapshot_changes' [feedback] to .memory\no_snapshot_changes.md
Saved memory 'legacy_dir_keep' [project] to .memory\legacy_dir_keep.md

--- MEMORY.md ---
# Memory Index

- legacy_dir_keep: Legacy directory must not be deleted [project]
- no_snapshot_changes: Do not modify test snapshots unless explicitly asked [feedback]
- prefer_powershell: User prefers PowerShell over bash/cmd [user]



In [132]:
# Load memory prompt (what gets injected into system prompt)
prompt = memory_mgr.load_memory_prompt()
print('Memory prompt section:')
print(prompt)

Memory prompt section:
# Memories (persistent across sessions)

## [user]
### prefer_powershell: User prefers PowerShell over bash/cmd
Always use PowerShell for system commands. Do not suggest bash or cmd alternatives.

## [feedback]
### no_snapshot_changes: Do not modify test snapshots unless explicitly asked
The user has corrected this twice. Test snapshots should only be updated when the user specifically requests it.

## [project]
### legacy_dir_keep: Legacy directory must not be deleted
The /legacy directory looks unused but deployment scripts still reference it. Do not delete or rename.



### Step 3: Register `save_memory` + `delete_memory` Tools

In [80]:
TOOLS = [
    {'type': 'function', 'function': {'name': 'powershell', 'description': 'Run a PowerShell command.', 'parameters': {'type': 'object', 'properties': {'command': {'type': 'string'}}, 'required': ['command']}}},
    {'type': 'function', 'function': {'name': 'read_file', 'description': 'Read file contents.', 'parameters': {'type': 'object', 'properties': {'path': {'type': 'string'}, 'limit': {'type': 'integer'}}, 'required': ['path']}}},
    {'type': 'function', 'function': {'name': 'write_file', 'description': 'Write content to a file.', 'parameters': {'type': 'object', 'properties': {'path': {'type': 'string'}, 'content': {'type': 'string'}}, 'required': ['path', 'content']}}},
    {'type': 'function', 'function': {'name': 'edit_file', 'description': 'Replace exact text in a file.', 'parameters': {'type': 'object', 'properties': {'path': {'type': 'string'}, 'old_text': {'type': 'string'}, 'new_text': {'type': 'string'}}, 'required': ['path', 'old_text', 'new_text']}}},
    {'type': 'function', 'function': {'name': 'save_memory', 'description': 'Save a persistent memory that survives across sessions. Types: user (preferences), feedback (corrections), project (non-obvious facts), reference (external URLs/pointers).', 'parameters': {'type': 'object', 'properties': {
        'name': {'type': 'string', 'description': 'Short identifier (e.g. prefer_tabs, db_schema)'},
        'description': {'type': 'string', 'description': 'One-line summary'},
        'type': {'type': 'string', 'enum': ['user', 'feedback', 'project', 'reference']},
        'content': {'type': 'string', 'description': 'Full memory content'},
    }, 'required': ['name', 'description', 'type', 'content']}}},
    {'type': 'function', 'function': {'name': 'delete_memory', 'description': 'Delete a memory by name.', 'parameters': {'type': 'object', 'properties': {
        'name': {'type': 'string', 'description': 'Name of the memory to delete'},
    }, 'required': ['name']}}},
]
TOOL_HANDLERS = {
    'powershell': run_powershell,
    'read_file':  lambda **kw: run_read(kw['path'], kw.get('limit')),
    'write_file': lambda **kw: run_write(kw['path'], kw['content']),
    'edit_file':  lambda **kw: run_edit(kw['path'], kw['old_text'], kw['new_text']),
    'save_memory': lambda **kw: memory_mgr.save_memory(kw['name'], kw['description'], kw['type'], kw['content']),
    'delete_memory': lambda **kw: memory_mgr.delete_memory(kw['name']),
}
print(f'Tools: {list(TOOL_HANDLERS.keys())}')

Tools: ['powershell', 'read_file', 'write_file', 'edit_file', 'save_memory', 'delete_memory']


### Step 4: Memory-Aware Agent Loop

The system prompt is **rebuilt each turn** so newly saved memories are visible immediately. Memory guidance tells the LLM when to save and when NOT to.

In [133]:
MEMORY_GUIDANCE = '''
When to save memories:
- User states a preference -> type: user
- User corrects you -> type: feedback
- You learn a project fact not obvious from code -> type: project
- You learn where an external resource lives -> type: reference

When NOT to save:
- Anything easily derivable from code (file structure, function names)
- Temporary task state (current branch, open PR numbers)
- Secrets or credentials
'''

def build_system_prompt() -> str:
    parts = [f'You are a coding agent at {WORKDIR}.']
    parts.append('You MUST use tools via function calling. Never output raw JSON.')
    memory_section = memory_mgr.load_memory_prompt()
    if memory_section:
        parts.append(memory_section)
    parts.append(MEMORY_GUIDANCE)
    return '\n\n'.join(parts)

def agent_loop(messages: list):
    turn = 0
    while True:
        turn += 1
        system = build_system_prompt()
        all_messages = [{'role': 'system', 'content': system}] + messages
        print(f"\n{'='*60}\nTurn {turn} ({len(all_messages)} msgs, {len(memory_mgr.memories)} memories)\n{'='*60}")
        for j, m in enumerate(all_messages):
            role = m['role']
            if role == 'system': print(f'  [{j}] system: ({len(system)} chars, includes memories)')
            elif role == 'user': print(f"  [{j}] user: {str(m.get('content',''))[:100]}")
            elif role == 'assistant':
                if m.get('tool_calls'): print(f"  [{j}] assistant: ({len(m['tool_calls'])} tool calls)")
                else: print(f"  [{j}] assistant: {str(m.get('content',''))[:100]}")
            elif role == 'tool': print(f"  [{j}] tool: {m['content'][:80]}")
        response = client.chat.completions.create(model=MODEL, messages=all_messages, tools=TOOLS)
        msg = response.choices[0].message
        print(f"\nTurn {turn} output: content={str(msg.content)[:80]}, tools={len(msg.tool_calls) if msg.tool_calls else 0}")
        assistant_msg = {'role': 'assistant', 'content': msg.content or ''}
        if msg.tool_calls:
            assistant_msg['tool_calls'] = [{'id': tc.id, 'type': 'function', 'function': {'name': tc.function.name, 'arguments': tc.function.arguments}} for tc in msg.tool_calls]
        messages.append(assistant_msg)
        if not msg.tool_calls:
            print(f'\n>>> Finished in {turn} turn(s)')
            return
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            handler = TOOL_HANDLERS.get(tc.function.name)
            try: output = handler(**args) if handler else f'Unknown: {tc.function.name}'
            except Exception as e: output = f'Error: {e}'
            print(f'  [{tc.function.name}] {str(output)[:200]}')
            messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': str(output)})
print('Memory-aware agent_loop ready')

Memory-aware agent_loop ready


## Try It!

1. `Remember that I prefer using pytest over unittest` (should save as `user` memory)
2. `Don't ever modify .env files without asking me first` (should save as `feedback`)
3. `The staging server URL is https://staging.example.com` (should save as `reference`)
4. Type `/memories` to see all saved memories
5. `What do you remember about my preferences?` (should recall from context)
6. `Delete the memory about pytest` (should use delete_memory tool)
7. Restart the kernel and run again — memories persist from `.memory/` files!

In [134]:
# Load memories at session start
memory_mgr.load_all()
mem_count = len(memory_mgr.memories)
if mem_count:
    print(f'[{mem_count} memories loaded into context]')
    for name, mem in memory_mgr.memories.items():
        print(f'  [{mem["type"]}] {name}: {mem["description"]}')
else:
    print('[No existing memories. The agent can create them with save_memory.]')

history = []
while True:
    try: query = input('s08 >> ')
    except (EOFError, KeyboardInterrupt): break
    if query.strip().lower() in ('q', 'exit', ''): break
    if query.strip() == '/memories':
        if memory_mgr.memories:
            for name, mem in memory_mgr.memories.items():
                print(f'  [{mem["type"]}] {name}: {mem["description"]}')
        else:
            print('  (no memories)')
        continue
    history.append({'role': 'user', 'content': query})
    agent_loop(history)
    print()

[Memory] Loaded 3 memories
[3 memories loaded into context]
  [project] legacy_dir_keep: Legacy directory must not be deleted
  [feedback] no_snapshot_changes: Do not modify test snapshots unless explicitly asked
  [user] prefer_powershell: User prefers PowerShell over bash/cmd


s08 >>  q


## What Changed From S07

| Component | Before (S07) | After (S10) |
|---|---|---|
| Cross-session state | None | File-based memory store (`.memory/`) |
| Memory types | None | user, feedback, project, reference |
| Storage format | None | YAML frontmatter markdown files |
| Session start | Cold start | Loads relevant memories into context |
| System prompt | Static | Rebuilt each turn with memory injection |
| Tools | 4 base | + save_memory, delete_memory |

> **Key takeaway**: Memory is a **curated store** of durable facts, not a dump of everything the agent has seen. The boundary between "save this" and "don't save this" is more important than the storage mechanism.

### Common Mistakes
1. **Storing things the repo can tell you** — stale copies that conflict with reality
2. **Storing live task progress** — "currently fixing auth" is task state, not memory
3. **Treating memory as absolute truth** — memory gives direction, current observation gives truth

---

**Congratulations! You've completed agent101!** 🎉

| Session | Concept |
|---|---|
| S01 | The Agent Loop |
| S02 | Multiple Tools |
| S03 | TodoWrite (Planning) |
| S04 | Subagents (Context Isolation) |
| S05 | Skill Loading (On-Demand Knowledge) |
| S06 | Context Compact (Compression) |
| S10 | Task System (Persistent Goals) |
| S11 | MCP & Plugins (External Integration) |
| S07 | Permission System (Safety Pipeline) |
| S08 | Memory System (Cross-Session Persistence) |

# S09: System Prompt Assembly

`s01 > s02 > s03 > s04 > s05 > s06 > s07 > s08 > [ s09 ] > s10 > s11`

> *"The system prompt is an assembly pipeline with clear sections and clear boundaries, not one big mysterious string."*
>
> **Harness layer**: Assembly -- turning scattered sources into one coherent model input.

When your agent had one tool and one job, a single hardcoded prompt string worked fine. But look at everything your harness has accumulated: a role description, tool definitions, loaded skills, saved memory, instruction files, and per-turn runtime context. If you keep cramming all of that into one big string, nobody can tell where each piece came from or how to change it safely.

## The Problem

You want to add a new tool. You open the system prompt, scroll past the role paragraph, past the safety rules, past the skill descriptions, past the memory block, and paste a tool description somewhere in the middle. A month later the prompt is 6,000 characters long, half is stale, and nobody remembers which lines change per turn vs. which stay fixed.

This is the natural trajectory of every agent that keeps its prompt in a single variable.

## The Solution: Prompt Assembly Pipeline

Each section has **one source** and **one responsibility**. A builder assembles them in a fixed order.

```
1. Core identity & rules        (stable)
2. Tool catalog                  (stable)
3. Skills                        (stable)
4. Memory                        (stable per session)
5. CLAUDE.md instruction chain   (stable)
--- DYNAMIC_BOUNDARY ---
6. Dynamic runtime context       (changes every turn)
```

### Stable vs Dynamic

| Type | Changes when? | Examples |
|---|---|---|
| **Stable** | Rarely or never during a session | Role, tools, safety rules, CLAUDE.md |
| **Dynamic** | Every turn or few turns | Date, cwd, current mode, reminders |

Separating them means the stable prefix can be **cached across turns** to save tokens.

### Step 1: The SystemPromptBuilder

Each `_build_*` method owns exactly one source. If you want to know where a line came from, check the one method responsible.

In [136]:
import datetime

DYNAMIC_BOUNDARY = '=== DYNAMIC_BOUNDARY ==='

class SystemPromptBuilder:
    def __init__(self, workdir: Path = None, tools: list = None):
        self.workdir = workdir or WORKDIR
        self.tools = tools or []
        self.skills_dir = self.workdir / 'skills'
        self.memory_dir = self.workdir / '.memory'

    # Section 1: Core identity
    def _build_core(self) -> str:
        return (
            f'You are a coding agent operating in {self.workdir}.\n'
            'You MUST use tools via function calling. Never output raw JSON.\n'
            'Always verify before assuming. Prefer reading files over guessing.'
        )

    # Section 2: Tool catalog
    def _build_tool_listing(self) -> str:
        if not self.tools: return ''
        lines = ['# Available tools']
        for t in self.tools:
            fn = t.get('function', t)
            name = fn.get('name', '')
            desc = fn.get('description', '')
            params = fn.get('parameters', {}).get('properties', {})
            plist = ', '.join(params.keys())
            lines.append(f'- {name}({plist}): {desc}')
        return '\n'.join(lines)

    # Section 3: Skill metadata (layer 1 from S05)
    def _build_skill_listing(self) -> str:
        if not self.skills_dir.exists(): return ''
        skills = []
        for skill_dir in sorted(self.skills_dir.iterdir()):
            skill_md = skill_dir / 'SKILL.md'
            if not skill_md.exists(): continue
            text = skill_md.read_text(encoding='utf-8')
            import re as _re
            match = _re.match(r'^---\s*\n(.*?)\n---', text, _re.DOTALL)
            if not match: continue
            meta = {}
            for line in match.group(1).splitlines():
                if ':' in line:
                    k, _, v = line.partition(':')
                    meta[k.strip()] = v.strip()
            skills.append(f"- {meta.get('name', skill_dir.name)}: {meta.get('description', '')}")
        if not skills: return ''
        return '# Available skills (use load_skill to load full content)\n' + '\n'.join(skills)

    # Section 4: Memory content
    def _build_memory_section(self) -> str:
        if not self.memory_dir.exists(): return ''
        memories = []
        for md_file in sorted(self.memory_dir.glob('*.md')):
            if md_file.name == 'MEMORY.md': continue
            text = md_file.read_text(encoding='utf-8')
            import re as _re
            match = _re.match(r'^---\s*\n(.*?)\n---\s*\n(.*)', text, _re.DOTALL)
            if not match: continue
            header, body = match.group(1), match.group(2).strip()
            meta = {}
            for line in header.splitlines():
                if ':' in line:
                    k, _, v = line.partition(':')
                    meta[k.strip()] = v.strip()
            memories.append(f"[{meta.get('type','project')}] {meta.get('name', md_file.stem)}: {meta.get('description','')}\n{body}")
        if not memories: return ''
        return '# Memories (persistent across sessions)\n\n' + '\n\n'.join(memories)

    # Section 5: CLAUDE.md instruction chain
    def _build_claude_md(self) -> str:
        sources = []
        user_claude = Path.home() / '.claude' / 'CLAUDE.md'
        if user_claude.exists():
            sources.append(('user global (~/.claude/CLAUDE.md)', user_claude.read_text(encoding='utf-8')))
        project_claude = self.workdir / 'CLAUDE.md'
        if project_claude.exists():
            sources.append(('project root (CLAUDE.md)', project_claude.read_text(encoding='utf-8')))
        if not sources: return ''
        parts = ['# CLAUDE.md instructions']
        for label, content in sources:
            parts.append(f'## From {label}')
            parts.append(content.strip())
        return '\n\n'.join(parts)

    # Section 6: Dynamic runtime context
    def _build_dynamic_context(self) -> str:
        lines = [
            f'Current date: {datetime.date.today().isoformat()}',
            f'Working directory: {self.workdir}',
            f'Model: {MODEL}',
            f'Platform: {os.name}',
        ]
        return '# Dynamic context\n' + '\n'.join(lines)

    # Assemble
    def build(self) -> str:
        sections = []
        for builder in [self._build_core, self._build_tool_listing,
                        self._build_skill_listing, self._build_memory_section,
                        self._build_claude_md]:
            s = builder()
            if s: sections.append(s)
        sections.append(DYNAMIC_BOUNDARY)
        dynamic = self._build_dynamic_context()
        if dynamic: sections.append(dynamic)
        return '\n\n'.join(sections)

print('SystemPromptBuilder class ready')

SystemPromptBuilder class ready


### Step 2: Per-Turn Reminders

Some info only matters for one turn (transient warnings, recovery hints). These go in a separate `system-reminder` message, not the main prompt.

In [137]:
def build_system_reminder(extra: str = None) -> dict | None:
    if not extra: return None
    return {'role': 'user', 'content': f'<system-reminder>\n{extra}\n</system-reminder>'}

# Example
reminder = build_system_reminder('The last tool call failed with a timeout. Try a simpler approach.')
print(reminder)

{'role': 'user', 'content': '<system-reminder>\nThe last tool call failed with a timeout. Try a simpler approach.\n</system-reminder>'}


### Step 3: Create a CLAUDE.md

CLAUDE.md files **layer** instructions -- multiple files contribute, later layers add to earlier ones rather than replacing them.

In [138]:
# Create a project-level CLAUDE.md
claude_md = WORKDIR / 'CLAUDE.md'
if not claude_md.exists():
    claude_md.write_text(
        '# Project Instructions\n\n'
        '- This is a teaching project. Keep code simple and well-commented.\n'
        '- Use PowerShell for all system commands (not bash/cmd).\n'
        '- When creating files, always use UTF-8 encoding.\n'
        '- Prefer explicit over implicit.\n',
        encoding='utf-8'
    )
    print(f'Created {claude_md}')
else:
    print(f'CLAUDE.md already exists: {claude_md.read_text(encoding="utf-8")[:200]}')

CLAUDE.md already exists: # Project Instructions

- This is a teaching project. Keep code simple and well-commented.
- Use PowerShell for all system commands (not bash/cmd).
- When creating files, always use UTF-8 encoding.
- 


### Step 4: Test the Builder

Let's see what the assembled prompt looks like with all our existing content (skills from S05, memories from S10, CLAUDE.md).

In [139]:
# Build with current tools
TOOLS = [
    {'type': 'function', 'function': {'name': 'powershell', 'description': 'Run a PowerShell command.', 'parameters': {'type': 'object', 'properties': {'command': {'type': 'string'}}, 'required': ['command']}}},
    {'type': 'function', 'function': {'name': 'read_file', 'description': 'Read file contents.', 'parameters': {'type': 'object', 'properties': {'path': {'type': 'string'}, 'limit': {'type': 'integer'}}, 'required': ['path']}}},
    {'type': 'function', 'function': {'name': 'write_file', 'description': 'Write content to a file.', 'parameters': {'type': 'object', 'properties': {'path': {'type': 'string'}, 'content': {'type': 'string'}}, 'required': ['path', 'content']}}},
    {'type': 'function', 'function': {'name': 'edit_file', 'description': 'Replace exact text in a file.', 'parameters': {'type': 'object', 'properties': {'path': {'type': 'string'}, 'old_text': {'type': 'string'}, 'new_text': {'type': 'string'}}, 'required': ['path', 'old_text', 'new_text']}}},
    {'type': 'function', 'function': {'name': 'save_memory', 'description': 'Save a persistent memory.', 'parameters': {'type': 'object', 'properties': {'name': {'type': 'string'}, 'description': {'type': 'string'}, 'type': {'type': 'string'}, 'content': {'type': 'string'}}, 'required': ['name', 'description', 'type', 'content']}}},
]
TOOL_HANDLERS = {
    'powershell': run_powershell,
    'read_file':  lambda **kw: run_read(kw['path'], kw.get('limit')),
    'write_file': lambda **kw: run_write(kw['path'], kw['content']),
    'edit_file':  lambda **kw: run_edit(kw['path'], kw['old_text'], kw['new_text']),
    'save_memory': lambda **kw: memory_mgr.save_memory(kw['name'], kw['description'], kw['type'], kw['content']),
}

builder = SystemPromptBuilder(workdir=WORKDIR, tools=TOOLS)
full_prompt = builder.build()

print(f'Total prompt length: {len(full_prompt)} chars (~{len(full_prompt)//4} tokens)')
print(f'\nSections found:')
for line in full_prompt.splitlines():
    if line.startswith('# ') or line == DYNAMIC_BOUNDARY:
        print(f'  {line}')
print(f'\n--- Full Prompt ---\n{full_prompt}\n--- End ---')

Total prompt length: 1644 chars (~411 tokens)

Sections found:
  # Available tools
  # Available skills (use load_skill to load full content)
  # Memories (persistent across sessions)
  # CLAUDE.md instructions
  # Project Instructions
  === DYNAMIC_BOUNDARY ===
  # Dynamic context

--- Full Prompt ---
You are a coding agent operating in C:\Users\concao\code\agent101.
You MUST use tools via function calling. Never output raw JSON.
Always verify before assuming. Prefer reading files over guessing.

# Available tools
- powershell(command): Run a PowerShell command.
- read_file(path, limit): Read file contents.
- write_file(path, content): Write content to a file.
- edit_file(path, old_text, new_text): Replace exact text in a file.
- save_memory(name, description, type, content): Save a persistent memory.

# Available skills (use load_skill to load full content)
- code-review: "Guidelines for reviewing Python code: style, bugs, security."

# Memories (persistent across sessions)

[project

### Step 5: Agent Loop with Assembled Prompt

The loop now uses the builder instead of a hardcoded string. The prompt is rebuilt each turn so new memories and dynamic context stay fresh.

In [141]:
def agent_loop(messages: list):
    turn = 0
    while True:
        turn += 1
        system = builder.build()
        all_messages = [{'role': 'system', 'content': system}] + messages
        print(f"\n{'='*60}\nTurn {turn} ({len(all_messages)} msgs, prompt={len(system)} chars)\n{'='*60}")
        for j, m in enumerate(all_messages):
            role = m['role']
            if role == 'system': print(f'  [{j}] system: ({len(system)} chars, assembled)')
            elif role == 'user': print(f"  [{j}] user: {str(m.get('content',''))[:100]}")
            elif role == 'assistant':
                if m.get('tool_calls'): print(f"  [{j}] assistant: ({len(m['tool_calls'])} tool calls)")
                else: print(f"  [{j}] assistant: {str(m.get('content',''))[:100]}")
            elif role == 'tool': print(f"  [{j}] tool: {m['content'][:80]}")
        response = client.chat.completions.create(model=MODEL, messages=all_messages, tools=TOOLS)
        msg = response.choices[0].message
        print(f"\nTurn {turn} output: content={str(msg.content)[:80]}, tools={len(msg.tool_calls) if msg.tool_calls else 0}")
        assistant_msg = {'role': 'assistant', 'content': msg.content or ''}
        if msg.tool_calls:
            assistant_msg['tool_calls'] = [{'id': tc.id, 'type': 'function', 'function': {'name': tc.function.name, 'arguments': tc.function.arguments}} for tc in msg.tool_calls]
        messages.append(assistant_msg)
        if not msg.tool_calls:
            print(f'\n>>> Finished in {turn} turn(s)')
            return
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            handler = TOOL_HANDLERS.get(tc.function.name)
            try: output = handler(**args) if handler else f'Unknown: {tc.function.name}'
            except Exception as e: output = f'Error: {e}'
            print(f'  [{tc.function.name}] {str(output)[:200]}')
            messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': str(output)})
print('Agent loop with assembled prompt ready')

Agent loop with assembled prompt ready


## Try It!

1. Type `/prompt` to see the full assembled system prompt
2. Type `/sections` to see just the section headers
3. `What tools do you have?` (info comes from the tool catalog section)
4. `What do you remember about me?` (info comes from the memory section)
5. `Save a memory that I prefer dark mode` (adds to memory, visible next turn)
6. Edit `CLAUDE.md` and notice it appears in the prompt on the next turn

In [88]:
history = []
while True:
    try: query = input('s09 >> ')
    except (EOFError, KeyboardInterrupt): break
    if query.strip().lower() in ('q', 'exit', ''): break
    if query.strip() == '/prompt':
        print('--- System Prompt ---')
        print(builder.build())
        print('--- End ---')
        continue
    if query.strip() == '/sections':
        for line in builder.build().splitlines():
            if line.startswith('# ') or line == DYNAMIC_BOUNDARY:
                print(f'  {line}')
        continue
    history.append({'role': 'user', 'content': query})
    agent_loop(history)
    print()

s09 >>  /prompt


--- System Prompt ---
You are a coding agent operating in C:\Users\concao\code\agent101.
You MUST use tools via function calling. Never output raw JSON.
Always verify before assuming. Prefer reading files over guessing.

# Available tools
- powershell(command): Run a PowerShell command.
- read_file(path, limit): Read file contents.
- write_file(path, content): Write content to a file.
- edit_file(path, old_text, new_text): Replace exact text in a file.
- save_memory(name, description, type, content): Save a persistent memory.

# Available skills (use load_skill to load full content)
- code-review: "Guidelines for reviewing Python code: style, bugs, security."

# Memories (persistent across sessions)

[project] legacy_dir_keep: Legacy directory must not be deleted
The /legacy directory looks unused but deployment scripts still reference it. Do not delete or rename.

[feedback] no_snapshot_changes: Do not modify test snapshots unless explicitly asked
The user has corrected this twice. Te

s09 >>  q


## What Changed From S08

| Component | Before (S08) | After (S11) |
|---|---|---|
| System prompt | Inline string + memory | Assembled pipeline (6 sections) |
| Stable vs dynamic | Mixed together | Separated by DYNAMIC_BOUNDARY |
| CLAUDE.md | Not supported | Layered instruction chain |
| Memory injection | Direct in prompt string | Via builder._build_memory_section() |
| Per-turn reminders | Mixed into prompt | Separate system-reminder message |
| Testability | Opaque string | Each section independently testable |

### Key Boundaries

| Source | Purpose | Changes |
|---|---|---|
| **Skills** | Optional capability packages loaded on demand | Rarely |
| **Memory** | Durable cross-session facts (user prefs, corrections) | Per session |
| **CLAUDE.md** | Standing instructions that layer without overwriting | Per project |
| **Dynamic context** | Date, cwd, mode, transient state | Every turn |

> **Key takeaway**: The system prompt is an assembly pipeline with clear sections and boundaries. Each section has one source, one purpose, and a predictable lifetime.

---

**Full course complete!**

| Session | Concept |
|---|---|
| S01 | The Agent Loop |
| S02 | Multiple Tools |
| S03 | TodoWrite (Planning) |
| S04 | Subagents (Context Isolation) |
| S05 | Skill Loading (On-Demand Knowledge) |
| S06 | Context Compact (Compression) |
| S10 | Task System (Persistent Goals) |
| S11 | MCP & Plugins (External Integration) |
| S07 | Permission System (Safety Pipeline) |
| S08 | Memory System (Cross-Session Persistence) |
| S09 | System Prompt Assembly (Pipeline Construction) |

# S10: Task System

`s01 > s02 > s03 > s04 > s05 > s06 > s07 > s08 > s09 > [ s10 ] > s11`

> *"Break big goals into small tasks, order them, persist to disk"*
>
> **Harness layer**: Persistent tasks — a file-based task graph that survives restarts.

## Problem

The S03 `TodoManager` is in-memory — restart the kernel and it's gone. It also has no concept of dependencies: task B should wait for task A, but the model has no way to express that. For complex projects with blocking dependencies, we need **persistent, structured tasks**.

## Solution

```
.tasks/
 +-- task_001.json   {"id": 1, "title": "...", "status": "done", "blockedBy": []}
 +-- task_002.json   {"id": 2, "title": "...", "status": "ready", "blockedBy": []}
 +-- task_003.json   {"id": 3, "title": "...", "status": "blocked", "blockedBy": [2]}

When task 2 completes -> auto-clear '2' from task 3's blockedBy -> task 3 becomes ready
```

Each task is a JSON file. Completing a task auto-clears it from dependents' `blockedBy` lists. Tasks survive kernel restarts.

### Step 1: TaskManager Class

File-based task management with dependency tracking. Each task gets its own JSON file in `.tasks/`.

In [89]:
class TaskManager:
    '''File-based task graph with dependencies.'''

    def __init__(self, tasks_dir=None):
        self.tasks_dir = Path(tasks_dir) if tasks_dir else WORKDIR / ".tasks"
        self.tasks_dir.mkdir(exist_ok=True)
        self._next_id = self._get_next_id()

    def _get_next_id(self) -> int:
        ids = []
        for f in self.tasks_dir.glob("task_*.json"):
            try:
                ids.append(int(f.stem.split("_")[1]))
            except (ValueError, IndexError):
                pass
        return max(ids, default=0) + 1

    def _task_path(self, task_id: int) -> Path:
        return self.tasks_dir / f"task_{task_id:03d}.json"

    def _read_task(self, task_id: int) -> dict:
        path = self._task_path(task_id)
        if not path.exists():
            return None
        return json.loads(path.read_text(encoding="utf-8"))

    def _write_task(self, task: dict):
        path = self._task_path(task["id"])
        path.write_text(json.dumps(task, indent=2), encoding="utf-8")

    def create(self, title: str, description: str = "", blocked_by: list = None) -> str:
        '''Create a new task. Returns confirmation.'''
        task = {
            "id": self._next_id,
            "title": title,
            "description": description,
            "status": "blocked" if blocked_by else "ready",
            "blockedBy": blocked_by or [],
        }
        self._write_task(task)
        self._next_id += 1
        return f"Created task #{task['id']}: {title} (status: {task['status']})"

    def get(self, task_id: int) -> str:
        '''Get task details.'''
        task = self._read_task(task_id)
        if not task:
            return f"Task #{task_id} not found."
        return json.dumps(task, indent=2)

    def update(self, task_id: int, status: str) -> str:
        '''Update task status. Completing a task clears dependencies.'''
        task = self._read_task(task_id)
        if not task:
            return f"Task #{task_id} not found."
        old_status = task["status"]
        task["status"] = status
        self._write_task(task)
        result = f"Task #{task_id}: {old_status} -> {status}"
        # Auto-clear from dependents when completed
        if status == "done":
            cleared = self._clear_dependency(task_id)
            if cleared:
                result += f" (unblocked: {cleared})"
        return result

    def _clear_dependency(self, completed_id: int) -> list:
        '''Remove completed_id from all dependents blockedBy lists.'''
        unblocked = []
        for f in self.tasks_dir.glob("task_*.json"):
            t = json.loads(f.read_text(encoding="utf-8"))
            if completed_id in t.get("blockedBy", []):
                t["blockedBy"].remove(completed_id)
                if not t["blockedBy"] and t["status"] == "blocked":
                    t["status"] = "ready"
                    unblocked.append(t["id"])
                self._write_task(t)
        return unblocked

    def list_all(self) -> str:
        '''List all tasks with their status.'''
        tasks = []
        for f in sorted(self.tasks_dir.glob("task_*.json")):
            tasks.append(json.loads(f.read_text(encoding="utf-8")))
        if not tasks:
            return "No tasks."
        lines = []
        icons = {"ready": "[ ]", "in_progress": "[>]", "done": "[x]", "blocked": "[!]"}
        for t in tasks:
            icon = icons.get(t['status'], '[?]')
            blocked = f" (blocked by: {t['blockedBy']})" if t.get('blockedBy') else ""
            lines.append(f"{icon} #{t['id']}: {t['title']}{blocked}")
        done = sum(1 for t in tasks if t['status'] == 'done')
        lines.append(f"\n({done}/{len(tasks)} done)")
        return "\n".join(lines)


TASKS = TaskManager()

# Quick test
print(TASKS.create("Set up project structure"))
print(TASKS.create("Write core module", blocked_by=[1]))
print(TASKS.create("Write tests", blocked_by=[2]))
print()
print(TASKS.list_all())
print()
print(TASKS.update(1, "done"))
print()
print(TASKS.list_all())

Created task #7: Set up project structure (status: ready)
Created task #8: Write core module (status: blocked)
Created task #9: Write tests (status: blocked)

[x] #1: Set up project structure
[ ] #2: Write core module
[!] #3: Write tests (blocked by: [2])
[ ] #4: Create Python calculator module
[!] #5: Add unit tests for calculator module (blocked by: [4])
[!] #6: Run calculator tests (blocked by: [5])
[ ] #7: Set up project structure
[!] #8: Write core module (blocked by: [1])
[!] #9: Write tests (blocked by: [2])

(1/9 done)

Task #1: done -> done (unblocked: [8])

[x] #1: Set up project structure
[ ] #2: Write core module
[!] #3: Write tests (blocked by: [2])
[ ] #4: Create Python calculator module
[!] #5: Add unit tests for calculator module (blocked by: [4])
[!] #6: Run calculator tests (blocked by: [5])
[ ] #7: Set up project structure
[ ] #8: Write core module
[!] #9: Write tests (blocked by: [2])

(1/9 done)


### Step 2: Task Tools & Updated Dispatch

Four new tools: `task_create`, `task_update`, `task_list`, `task_get`. These give the model full control over the persistent task graph.

In [90]:
# Reset TaskManager for the session
TASKS = TaskManager()

TASK_MGMT_TOOLS = [
    {"type": "function", "function": {
        "name": "task_create",
        "description": "Create a persistent task. Use blocked_by for dependencies.",
        "parameters": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "description": {"type": "string"},
                "blocked_by": {"type": "array", "items": {"type": "integer"}, "description": "Task IDs this depends on."},
            },
            "required": ["title"],
        },
    }},
    {"type": "function", "function": {
        "name": "task_update",
        "description": "Update task status (ready, in_progress, done). Completing auto-unblocks dependents.",
        "parameters": {
            "type": "object",
            "properties": {
                "task_id": {"type": "integer"},
                "status": {"type": "string", "enum": ["ready", "in_progress", "done"]},
            },
            "required": ["task_id", "status"],
        },
    }},
    {"type": "function", "function": {
        "name": "task_list",
        "description": "List all tasks with status and dependencies.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "task_get",
        "description": "Get full details of a specific task.",
        "parameters": {
            "type": "object",
            "properties": {"task_id": {"type": "integer"}},
            "required": ["task_id"],
        },
    }},
]

TOOLS = CHILD_TOOLS + [
    TASK_TOOL,
    {"type": "function", "function": {
        "name": "load_skill",
        "description": "Load full content of a skill by name.",
        "parameters": {"type": "object", "properties": {"name": {"type": "string"}}, "required": ["name"]},
    }},
    {"type": "function", "function": {
        "name": "compact",
        "description": "Compress conversation context.",
        "parameters": {"type": "object", "properties": {}},
    }},
] + TASK_MGMT_TOOLS

TOOL_HANDLERS = {
    **CHILD_HANDLERS,
    "task":        lambda **kw: run_subagent(kw["prompt"]),
    "load_skill":  lambda **kw: SKILLS.get_content(kw["name"]),
    "compact":     run_compact,
    "task_create": lambda **kw: TASKS.create(kw["title"], kw.get("description", ""), kw.get("blocked_by")),
    "task_update": lambda **kw: TASKS.update(kw["task_id"], kw["status"]),
    "task_list":   lambda **kw: TASKS.list_all(),
    "task_get":    lambda **kw: TASKS.get(kw["task_id"]),
}

SYSTEM = (
    f"You are a coding agent at {WORKDIR}.\n"
    "Use task_create/task_update/task_list to manage persistent tasks with dependencies.\n"
    "Set status to in_progress before starting, done when complete.\n"
    "Completing a task auto-unblocks dependents. Prefer tools over prose.\n\n"
    f"{SKILLS.get_descriptions()}"
)

TODO = TodoManager()
print(f"Tools ({len(TOOLS)}): {[t['function']['name'] for t in TOOLS]}")

Tools (12): ['powershell', 'read_file', 'write_file', 'edit_file', 'todo', 'task', 'load_skill', 'compact', 'task_create', 'task_update', 'task_list', 'task_get']


### Step 3: Agent Loop

Same loop with all previous features (compaction, nag, skill loading). Task management works through the dispatch map.

In [ ]:
def agent_loop(messages: list):
    '''Agent loop with persistent task system.'''
    turn = 0
    rounds_since_todo = 0
    compact_requested = False
    while True:
        turn += 1

        # Compaction layers
        messages[:] = micro_compact(messages)
        est = estimate_tokens(messages)
        if est > TOKEN_THRESHOLD or compact_requested:
            print(f"  [compact] Auto-compacting...")
            messages[:] = auto_compact(messages)
            compact_requested = False

        all_messages = [{"role": "system", "content": SYSTEM}] + messages

        if rounds_since_todo >= 3:
            all_messages.append({"role": "user", "content": "<reminder>Update your todos/tasks.</reminder>"})
            print(f"  [nag] Injected task reminder")

        print(f"\n{'='*60}")
        print(f"Turn {turn} - LLM Input ({len(all_messages)} msgs, ~{estimate_tokens(all_messages)} tokens):")
        print(f"{'='*60}")
        for j, m in enumerate(all_messages):
            role = m["role"]
            if role == "system":
                print(f"  [{j}] system: {m['content'][:80]}...")
            elif role == "user":
                content = m.get("content", "")
                if isinstance(content, str):
                    print(f"  [{j}] user: {content[:100]}")
                else:
                    print(f"  [{j}] user: (tool results)")
            elif role == "assistant":
                tc = m.get("tool_calls")
                if tc:
                    print(f"  [{j}] assistant: (called {len(tc)} tool(s))")
                else:
                    print(f"  [{j}] assistant: {str(m.get('content', ''))[:100]}")
            elif role == "tool":
                print(f"  [{j}] tool: {m['content'][:80]}")

        response = client.chat.completions.create(
            model=MODEL,
            messages=all_messages,
            tools=TOOLS,
        )
        msg = response.choices[0].message

        print(f"\nTurn {turn} - LLM Output:")
        print(f"  content:    {msg.content}")
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  tool_call:  {tc.function.name}({tc.function.arguments[:100]})")
        else:
            print(f"  tool_calls: None (done!)")

        assistant_msg = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(assistant_msg)

        if not msg.tool_calls:
            print(f"\n>>> Agent finished in {turn} turn(s)")
            print(f"\n--- Task Board ---")
            print(TASKS.list_all())
            return

        used_todo = False
        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            handler = TOOL_HANDLERS.get(name)
            try:
                output = handler(**args) if handler else f"Unknown tool: {name}"
            except Exception as e:
                output = f"Error: {e}"
            print(f"\n  [exec] {name}: {json.dumps(args)[:100]}")
            print(f"  [result] {str(output)[:300]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(output),
            })
            if name in ("todo", "task_create", "task_update", "task_list"):
                used_todo = True
            if name == "compact":
                compact_requested = True

        rounds_since_todo = 0 if used_todo else rounds_since_todo + 1

## Try It!

Try a project that needs dependencies:
1. `Build a Python calculator package: create the module, add tests, then run them`
2. `List tasks` (see the task board)

Watch for:
- `blockedBy` arrays being set on dependent tasks
- Auto-unblocking when a task completes
- Tasks surviving in `.tasks/` directory (check with `ls .tasks/`)

In [35]:
# Interactive REPL
TASKS = TaskManager()
TODO = TodoManager()
history = []
while True:
    try:
        query = input("s07 >> ")
    except (EOFError, KeyboardInterrupt):
        break
    if query.strip().lower() in ("q", "exit", ""):
        break
    history.append({"role": "user", "content": query})
    agent_loop(history)
    print()

s07 >>  Build a Python calculator package: create the module, add tests, then run them



Turn 1 - LLM Input (2 msgs, ~140 tokens):
  [0] system: You are a coding agent at C:\Users\concao\code\agent101.
Use task_create/task_up...
  [1] user: Build a Python calculator package: create the module, add tests, then run them

Turn 1 - LLM Output:
  content:    None
  tool_call:  task_create({"title":"Create Python calculator module","description":"Implement a Python module supporting basic)

  [exec] task_create: {"title": "Create Python calculator module", "description": "Implement a Python module supporting ba
  [result] Created task #4: Create Python calculator module (status: ready)

Turn 2 - LLM Input (4 msgs, ~252 tokens):
  [0] system: You are a coding agent at C:\Users\concao\code\agent101.
Use task_create/task_up...
  [1] user: Build a Python calculator package: create the module, add tests, then run them
  [2] assistant: (called 1 tool(s))
  [3] tool: Created task #4: Create Python calculator module (status: ready)

Turn 2 - LLM Output:
  content:    None
  tool_call: 

s07 >>  q


## What Changed From S06

| Component      | Before (S06)           | After (S07)                              |
|----------------|------------------------|------------------------------------------|
| Tools          | 8                      | 12 (+task_create/update/list/get)        |
| Planning       | In-memory TodoManager  | + file-based TaskManager                 |
| Dependencies   | None                   | `blockedBy` arrays, auto-unblocking      |
| Persistence    | Lost on restart        | JSON files in `.tasks/` survive restarts |

> **Key insight**: File-based tasks persist across sessions. Dependencies create a DAG — the model can only work on unblocked tasks, enforcing correct ordering.

---

**Next: Session 08 — Background Tasks** → run slow operations in the background; the agent keeps thinking.

# S11: MCP & Plugin System

`s01 > s02 > s03 > s04 > s05 > s06 > s07 > s08 > s09 > s10 > [ s11 ]`

> *"External tools should enter the same tool pipeline, not form a completely separate world."*
>
> **Harness layer**: Integration — tools aren't just in your code.

Up to this point, every tool your agent uses — powershell, read, write, edit, tasks — lives inside your Python harness. You wrote each one by hand. That works for a teaching codebase, but a real agent needs to talk to databases, browsers, cloud services, and tools that **don't exist yet**. Hard-coding every capability is not sustainable.

## The Problem

Your agent's capabilities are frozen at build time. Want to query Postgres? Write a new Python handler. Control a browser? Another handler. Every new capability means changing the core harness, re-testing, and redeploying.

Meanwhile, other teams build specialized servers that already know how to talk to these systems. You need a **standard protocol** so those servers can expose tools to your agent — and your agent can call them as naturally as its native tools.

## The Solution: MCP (Model Context Protocol)

MCP gives your agent a standard way to connect to external capability servers over **stdio**. The agent starts a server process, asks what tools it provides, normalizes their names with a prefix, and routes calls to that server — all through the **same tool pipeline** that handles native tools.

```
LLM
  |
  | asks to call a tool
  v
Agent tool router
  |
  +-- native tool  -> local Python handler
  |
  +-- MCP tool     -> external MCP server
                        |
                        v
                    return result
```

### Plugin vs Server vs Tool

| Layer | What it is | What it's for |
|---|---|---|
| **Plugin manifest** | A config declaration | Tells the harness which servers to discover and launch |
| **MCP server** | An external process | Exposes a set of capabilities |
| **MCP tool** | One callable capability | The concrete thing the model invokes |

Shortest memory aid: **plugin = discovery, server = connection, tool = invocation**

### Step 1: Build an MCP Client

The `MCPClient` manages a connection to one external server over stdio. It starts the process, sends a JSON-RPC handshake, and caches available tools.

In [142]:
class MCPClient:
    def __init__(self, server_name: str, command: str, args: list = None, env: dict = None):
        self.server_name = server_name
        self.command = command
        self.args = args or []
        self.env = {**os.environ, **(env or {})}
        self.process = None
        self._request_id = 0
        self._tools = []

    def connect(self):
        try:
            self.process = subprocess.Popen(
                [self.command] + self.args,
                stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                stderr=subprocess.PIPE, env=self.env, text=True,
            )
            self._send({'method': 'initialize', 'params': {
                'protocolVersion': '2024-11-05',
                'capabilities': {},
                'clientInfo': {'name': 'agent101', 'version': '1.0'},
            }})
            response = self._recv()
            if response and 'result' in response:
                self._send({'method': 'notifications/initialized'})
                return True
        except FileNotFoundError:
            print(f'[MCP] Server command not found: {self.command}')
        except Exception as e:
            print(f'[MCP] Connection failed: {e}')
        return False

    def list_tools(self) -> list:
        self._send({'method': 'tools/list', 'params': {}})
        response = self._recv()
        if response and 'result' in response:
            self._tools = response['result'].get('tools', [])
        return self._tools

    def call_tool(self, tool_name: str, arguments: dict) -> str:
        self._send({'method': 'tools/call', 'params': {'name': tool_name, 'arguments': arguments}})
        response = self._recv()
        if response and 'result' in response:
            content = response['result'].get('content', [])
            return '\n'.join(c.get('text', str(c)) for c in content)
        if response and 'error' in response:
            return f"MCP Error: {response['error'].get('message', 'unknown')}"
        return 'MCP Error: no response'

    def get_agent_tools(self) -> list:
        """Convert MCP tools to Azure OpenAI function format with mcp__{server}__{tool} prefix."""
        agent_tools = []
        for tool in self._tools:
            prefixed = f'mcp__{self.server_name}__{tool["name"]}'
            schema = tool.get('inputSchema', {'type': 'object', 'properties': {}})
            agent_tools.append({'type': 'function', 'function': {
                'name': prefixed,
                'description': tool.get('description', ''),
                'parameters': schema,
            }})
        return agent_tools

    def disconnect(self):
        if self.process:
            try:
                self._send({'method': 'shutdown'})
                self.process.terminate()
                self.process.wait(timeout=5)
            except Exception:
                self.process.kill()
            self.process = None

    def _send(self, message: dict):
        if not self.process or self.process.poll() is not None: return
        self._request_id += 1
        envelope = {'jsonrpc': '2.0', 'id': self._request_id, **message}
        try:
            self.process.stdin.write(json.dumps(envelope) + '\n')
            self.process.stdin.flush()
        except (BrokenPipeError, OSError): pass

    def _recv(self) -> dict | None:
        if not self.process or self.process.poll() is not None: return None
        try:
            line = self.process.stdout.readline()
            if line: return json.loads(line)
        except (json.JSONDecodeError, OSError): pass
        return None

print('MCPClient class ready')

MCPClient class ready


### Step 2: Plugin Discovery

If MCP answers *"how does the agent talk to an external server,"* plugins answer *"how are those servers discovered?"*

A minimal plugin is a `.agent-plugin/plugin.json` manifest:

```json
{
  "name": "my-db-tools",
  "mcpServers": {
    "postgres": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-postgres"]
    }
  }
}
```

In [143]:
class PluginLoader:
    def __init__(self, search_dirs: list = None):
        self.search_dirs = search_dirs or [WORKDIR]
        self.plugins = {}

    def scan(self) -> list:
        found = []
        for d in self.search_dirs:
            manifest = Path(d) / '.agent-plugin' / 'plugin.json'
            if manifest.exists():
                try:
                    data = json.loads(manifest.read_text(encoding='utf-8'))
                    name = data.get('name', manifest.parent.parent.name)
                    self.plugins[name] = data
                    found.append(name)
                except Exception as e:
                    print(f'[Plugin] Failed: {manifest}: {e}')
        return found

    def get_mcp_servers(self) -> dict:
        servers = {}
        for pname, manifest in self.plugins.items():
            for sname, config in manifest.get('mcpServers', {}).items():
                servers[f'{pname}__{sname}'] = config
        return servers

print('PluginLoader class ready')

PluginLoader class ready


### Step 3: Unified Tool Router

The router doesn't care whether a tool is native or external. If the name starts with `mcp__`, route to the MCP server; otherwise call the local handler.

```
mcp__postgres__query   ->  split on '__'  ->  server='postgres', tool='query'
powershell             ->  not MCP        ->  local handler
```

In [144]:
class MCPToolRouter:
    def __init__(self):
        self.clients = {}

    def register_client(self, client: MCPClient):
        self.clients[client.server_name] = client

    def is_mcp_tool(self, name: str) -> bool:
        return name.startswith('mcp__')

    def call(self, tool_name: str, arguments: dict) -> str:
        parts = tool_name.split('__', 2)
        if len(parts) != 3:
            return f'Error: Invalid MCP tool name: {tool_name}'
        _, server_name, actual_tool = parts
        client = self.clients.get(server_name)
        if not client:
            return f'Error: MCP server not found: {server_name}'
        return client.call_tool(actual_tool, arguments)

    def get_all_tools(self) -> list:
        tools = []
        for c in self.clients.values():
            tools.extend(c.get_agent_tools())
        return tools

mcp_router = MCPToolRouter()
print('MCPToolRouter ready')

MCPToolRouter ready


### Step 4: Build the Unified Tool Pool

Native tools + MCP tools merge into one flat list. The agent loop doesn't know the difference — it just sees tools.

In [145]:
# Native tools (carried from previous sessions)
NATIVE_TOOLS = [
    {'type': 'function', 'function': {'name': 'powershell', 'description': 'Run a PowerShell command.', 'parameters': {'type': 'object', 'properties': {'command': {'type': 'string'}}, 'required': ['command']}}},
    {'type': 'function', 'function': {'name': 'read_file', 'description': 'Read file contents.', 'parameters': {'type': 'object', 'properties': {'path': {'type': 'string'}, 'limit': {'type': 'integer'}}, 'required': ['path']}}},
    {'type': 'function', 'function': {'name': 'write_file', 'description': 'Write content to a file.', 'parameters': {'type': 'object', 'properties': {'path': {'type': 'string'}, 'content': {'type': 'string'}}, 'required': ['path', 'content']}}},
    {'type': 'function', 'function': {'name': 'edit_file', 'description': 'Replace exact text in a file.', 'parameters': {'type': 'object', 'properties': {'path': {'type': 'string'}, 'old_text': {'type': 'string'}, 'new_text': {'type': 'string'}}, 'required': ['path', 'old_text', 'new_text']}}},
]
NATIVE_HANDLERS = {
    'powershell': run_powershell,
    'read_file':  lambda **kw: run_read(kw['path'], kw.get('limit')),
    'write_file': lambda **kw: run_write(kw['path'], kw['content']),
    'edit_file':  lambda **kw: run_edit(kw['path'], kw['old_text'], kw['new_text']),
}

def build_tool_pool() -> tuple:
    """Assemble native + MCP tools. Native takes precedence on name conflicts."""
    all_tools = list(NATIVE_TOOLS)
    all_handlers = dict(NATIVE_HANDLERS)
    mcp_tools = mcp_router.get_all_tools()
    native_names = {t['function']['name'] for t in all_tools}
    for tool in mcp_tools:
        name = tool['function']['name']
        if name not in native_names:
            all_tools.append(tool)
            all_handlers[name] = lambda tool_name=name, **kw: mcp_router.call(tool_name, kw)
    return all_tools, all_handlers

tools, handlers = build_tool_pool()
print(f'Tool pool: {len(tools)} tools ({len(mcp_router.get_all_tools())} from MCP)')

Tool pool: 4 tools (0 from MCP)


### Step 5: Permission Gate

**Most important rule**: external tools pass through the **same permission checks** as native tools. If MCP tools bypass permissions, you've created a security backdoor.

In [146]:
PERMISSION_MODES = ('default', 'auto')
READ_PREFIXES = ('read', 'list', 'get', 'show', 'search', 'query', 'inspect')
HIGH_RISK_PREFIXES = ('delete', 'remove', 'drop', 'shutdown')

class PermissionGate:
    def __init__(self, mode='default'):
        self.mode = mode if mode in PERMISSION_MODES else 'default'

    def classify(self, tool_name: str, tool_input: dict) -> dict:
        if tool_name.startswith('mcp__'):
            _, server, actual = tool_name.split('__', 2)
            source = 'mcp'
        else:
            server, actual, source = None, tool_name, 'native'
        low = actual.lower()
        if actual == 'read_file' or low.startswith(READ_PREFIXES):
            risk = 'read'
        elif actual == 'powershell':
            cmd = tool_input.get('command', '')
            risk = 'high' if any(d in cmd for d in DANGEROUS_COMMANDS) else 'write'
        elif low.startswith(HIGH_RISK_PREFIXES):
            risk = 'high'
        else:
            risk = 'write'
        return {'source': source, 'server': server, 'tool': actual, 'risk': risk}

    def check(self, tool_name: str, tool_input: dict) -> dict:
        intent = self.classify(tool_name, tool_input)
        if intent['risk'] == 'read':
            return {'allow': True, 'reason': 'Read-only', 'intent': intent}
        if self.mode == 'auto' and intent['risk'] != 'high':
            return {'allow': True, 'reason': 'Auto mode', 'intent': intent}
        if intent['risk'] == 'high':
            label = f"{intent['source']}:{intent.get('server','')}/{intent['tool']}" if intent['server'] else f"{intent['source']}:{intent['tool']}"
            preview = json.dumps(tool_input)[:200]
            print(f"\n  [Permission] {label} risk={intent['risk']}: {preview}")
            try:
                ans = input('  Allow? (y/n): ').strip().lower()
            except (EOFError, KeyboardInterrupt):
                ans = 'n'
            return {'allow': ans in ('y','yes'), 'reason': 'User decision', 'intent': intent}
        return {'allow': True, 'reason': 'Default allow write', 'intent': intent}

perm_gate = PermissionGate(mode='auto')
print('PermissionGate ready (mode: auto)')

PermissionGate ready (mode: auto)


### Step 6: Agent Loop with Unified Tools

The loop is almost unchanged from S07 — it just uses `build_tool_pool()` and routes through the permission gate.

In [147]:
SYSTEM = f"""You are a coding agent at {WORKDIR}.
You MUST use tools via function calling. Never output raw JSON.
You have both native tools and MCP tools available.
MCP tools are prefixed with mcp__{{server}}__{{tool}}.
All tools pass through the same permission gate before execution."""

def agent_loop(messages: list):
    tools, handlers = build_tool_pool()
    turn = 0
    while True:
        turn += 1
        all_messages = [{'role': 'system', 'content': SYSTEM}] + messages
        print(f"\n{'='*60}\nTurn {turn} ({len(all_messages)} msgs, {len(tools)} tools)\n{'='*60}")
        for j, m in enumerate(all_messages):
            role = m['role']
            if role == 'system': print(f'  [{j}] system: ...')
            elif role == 'user': print(f"  [{j}] user: {str(m.get('content',''))[:100]}")
            elif role == 'assistant':
                if m.get('tool_calls'): print(f"  [{j}] assistant: ({len(m['tool_calls'])} tool calls)")
                else: print(f"  [{j}] assistant: {str(m.get('content',''))[:100]}")
            elif role == 'tool': print(f"  [{j}] tool: {m['content'][:80]}")
        response = client.chat.completions.create(model=MODEL, messages=all_messages, tools=tools)
        msg = response.choices[0].message
        print(f"\nTurn {turn} output: content={str(msg.content)[:80]}, tools={len(msg.tool_calls) if msg.tool_calls else 0}")
        assistant_msg = {'role': 'assistant', 'content': msg.content or ''}
        if msg.tool_calls:
            assistant_msg['tool_calls'] = [{'id': tc.id, 'type': 'function', 'function': {'name': tc.function.name, 'arguments': tc.function.arguments}} for tc in msg.tool_calls]
        messages.append(assistant_msg)
        if not msg.tool_calls:
            print(f'\n>>> Finished in {turn} turn(s)')
            return
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            decision = perm_gate.check(tc.function.name, args)
            if not decision['allow']:
                output = f"Permission denied: {decision['reason']}"
            else:
                handler = handlers.get(tc.function.name)
                if handler:
                    try: output = handler(**args)
                    except Exception as e: output = f'Error: {e}'
                elif mcp_router.is_mcp_tool(tc.function.name):
                    try: output = mcp_router.call(tc.function.name, args)
                    except Exception as e: output = f'MCP Error: {e}'
                else:
                    output = f'Unknown tool: {tc.function.name}'
            src = decision['intent']['source']
            print(f'  [{src}] {tc.function.name}: {str(output)[:200]}')
            messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': str(output)})
print('agent_loop ready with unified tool routing')

agent_loop ready with unified tool routing


### Step 7: Create a Demo MCP Server

Let's create a tiny MCP server in Python that exposes two tools: `get_time` and `echo`. This demonstrates the full round-trip without needing external packages.

In [148]:
# Create a minimal MCP server for testing
mcp_server_code = r'''
import json, sys, datetime

TOOLS = [
    {"name": "get_time", "description": "Get the current date and time.", "inputSchema": {"type": "object", "properties": {}}},
    {"name": "echo", "description": "Echo back the input message.", "inputSchema": {"type": "object", "properties": {"message": {"type": "string"}}, "required": ["message"]}},
]

def handle(request):
    method = request.get("method", "")
    rid = request.get("id")
    if method == "initialize":
        return {"jsonrpc": "2.0", "id": rid, "result": {"protocolVersion": "2024-11-05", "capabilities": {"tools": {}}, "serverInfo": {"name": "demo-server", "version": "1.0"}}}
    if method == "notifications/initialized":
        return None
    if method == "tools/list":
        return {"jsonrpc": "2.0", "id": rid, "result": {"tools": TOOLS}}
    if method == "tools/call":
        name = request.get("params", {}).get("name", "")
        args = request.get("params", {}).get("arguments", {})
        if name == "get_time":
            text = datetime.datetime.now().isoformat()
        elif name == "echo":
            text = f"Echo: {args.get('message', '')}" 
        else:
            return {"jsonrpc": "2.0", "id": rid, "error": {"code": -32601, "message": f"Unknown tool: {name}"}}
        return {"jsonrpc": "2.0", "id": rid, "result": {"content": [{"type": "text", "text": text}]}}
    if method == "shutdown":
        return {"jsonrpc": "2.0", "id": rid, "result": {}}
    return {"jsonrpc": "2.0", "id": rid, "error": {"code": -32601, "message": f"Unknown method: {method}"}}

for line in sys.stdin:
    line = line.strip()
    if not line: continue
    try:
        req = json.loads(line)
        resp = handle(req)
        if resp:
            sys.stdout.write(json.dumps(resp) + "\n")
            sys.stdout.flush()
    except Exception as e:
        sys.stderr.write(f"Server error: {e}\n")
'''

server_path = WORKDIR / 'mcp_demo_server.py'
server_path.write_text(mcp_server_code.strip(), encoding='utf-8')
print(f'Created demo MCP server at: {server_path}')

Created demo MCP server at: C:\Users\concao\code\agent101\mcp_demo_server.py


### Step 8: Connect and Test!

Now let's connect our agent to the demo MCP server and see the full round-trip:
1. Start the server process
2. Handshake (initialize)
3. List tools
4. Tools appear in the unified pool with `mcp__demo__` prefix

In [149]:
# Connect to our demo MCP server
demo_client = MCPClient('demo', 'python', [str(WORKDIR / 'mcp_demo_server.py')])
if demo_client.connect():
    tools_found = demo_client.list_tools()
    print(f'Connected! Found {len(tools_found)} tools:')
    for t in tools_found:
        print(f"  - {t['name']}: {t.get('description','')}")
    mcp_router.register_client(demo_client)
    # Rebuild tool pool
    tools, handlers = build_tool_pool()
    print(f'\nUnified tool pool: {len(tools)} tools')
    for t in tools:
        prefix = '[MCP] ' if t['function']['name'].startswith('mcp__') else '       '
        print(f'  {prefix}{t["function"]["name"]}')
else:
    print('Failed to connect to demo server')

Connected! Found 2 tools:
  - get_time: Get the current date and time.
  - echo: Echo back the input message.

Unified tool pool: 6 tools
         powershell
         read_file
         write_file
         edit_file
  [MCP] mcp__demo__get_time
  [MCP] mcp__demo__echo


### Quick Test: Call MCP tools directly

In [150]:
# Test MCP tools directly (before going through the LLM)
print('Direct MCP call - get_time:')
print(f'  {mcp_router.call("mcp__demo__get_time", {})}')
print('\nDirect MCP call - echo:')
print(f'  {mcp_router.call("mcp__demo__echo", {"message": "Hello from agent101!"})}')

Direct MCP call - get_time:
  2026-04-17T15:02:54.180003

Direct MCP call - echo:
  Echo: Hello from agent101!


## Try It!

The agent now has both native and MCP tools. Try:

1. `What time is it?` (should use mcp__demo__get_time)
2. `Echo the message: MCP is working!` (should use mcp__demo__echo)
3. `List files in the current directory` (should use native powershell)
4. `What time is it, and also list files in this folder` (should mix native + MCP)
5. Type `/tools` to see the full tool pool
6. Type `/mcp` to see connected MCP servers

In [151]:
history = []
while True:
    try: query = input('s11 >> ')
    except (EOFError, KeyboardInterrupt): break
    if query.strip().lower() in ('q', 'exit', ''): break
    if query.strip() == '/tools':
        tools, _ = build_tool_pool()
        for t in tools:
            prefix = '[MCP] ' if t['function']['name'].startswith('mcp__') else '       '
            print(f"  {prefix}{t['function']['name']}: {t['function'].get('description','')[:60]}")
        continue
    if query.strip() == '/mcp':
        if mcp_router.clients:
            for name, c in mcp_router.clients.items():
                print(f'  {name}: {len(c.get_agent_tools())} tools')
        else:
            print('  (no MCP servers connected)')
        continue
    history.append({'role': 'user', 'content': query})
    agent_loop(history)
    print()

s11 >>  What time is it?



Turn 1 (2 msgs, 6 tools)
  [0] system: ...
  [1] user: What time is it?

Turn 1 output: content=None, tools=1
  [mcp] mcp__demo__get_time: 2026-04-17T15:03:08.309376

Turn 2 (4 msgs, 6 tools)
  [0] system: ...
  [1] user: What time is it?
  [2] assistant: (1 tool calls)
  [3] tool: 2026-04-17T15:03:08.309376

Turn 2 output: content=The current time is 3:03 PM on April 17, 2026., tools=0

>>> Finished in 2 turn(s)



s11 >>  q


In [ ]:
# Cleanup: disconnect MCP servers
for c in mcp_router.clients.values():
    c.disconnect()
print('MCP servers disconnected')

## How It Plugs Into The Full Harness

```
startup
  -> plugin loader finds manifests
  -> server configs are extracted
  -> MCP clients connect and list tools
  -> external tools normalized into the same pool

runtime
  -> LLM emits tool call
  -> shared permission gate
  -> native route or MCP route
  -> result returns to the same loop
```

Different entry point, same control plane.

## What Changed From S10

| Component | Before (S10) | After (S08) |
|---|---|---|
| Tool sources | All native (local Python) | Native + external MCP servers |
| Tool naming | Flat names (powershell, read_file) | + prefixed externals (mcp__demo__get_time) |
| Routing | Single handler map | Unified router: native + MCP dispatch |
| Capability growth | Edit harness code for each tool | Add plugin manifest or connect a server |
| Permission scope | Native tools only | Native + external through same gate |

> **Key insight**: External capabilities should enter the same tool pipeline as native ones — same naming, same routing, same permissions — so the agent loop never needs to know the difference.

---

**Congratulations!** You've built a mini coding agent from scratch:

| Session | Concept |
|---|---|
| S01 | The Agent Loop |
| S02 | Multiple Tools |
| S03 | TodoWrite (Planning) |
| S04 | Subagents (Context Isolation) |
| S05 | Skill Loading (On-Demand Knowledge) |
| S06 | Context Compact (Compression) |
| S10 | Task System (Persistent Goals) |
| S11 | MCP & Plugins (External Integration) |